# Table of Contents

- [NGC 2477 Open Cluster Age and Distance](#ngc-2477-open-cluster-age-and-distance)
  - [1. Import libraries and Setup Environement](#1-import-libraries-and-setup-environement)
  - [2. FITS loading and reusable star detector](#2-fits-loading-and-reusable-star-detector)
  - [3. Detect candidate stars](#3-detect-candidate-stars)
  - [4. Estimate FWHM for each detected star](#4-estimate-fwhm-for-each-detected-star)
  - [5. Add quality measurements](#5-add-quality-measurements)
  - [6. Apply filters](#6-apply-filters)
  - [7. Save the quality-controlled star catalog](#7-save-the-quality-controlled-star-catalog)
  - [8. Automated V-Band Calibration Using Gaia DR3 Synthetic Photometry](#8-automated-v-band-calibration-using-gaia-dr3-synthetic-photometry)
    - [8.1 Obtain a WCS solution (pixel position → RA/Dec)](#81-obtain-a-wcs-solution-pixel-position-radec)
    - [8.2 Query Gaia DR3 synthetic B/V/R reference magnitudes](#82-query-gaia-dr3-synthetic-bvr-reference-magnitudes)
    - [8.3 Match detected stars to Gaia (calibration only)](#83-match-detected-stars-to-gaia-calibration-only)
    - [8.4 Reusable calibration helpers: photometry → references → model](#84-reusable-calibration-helpers-photometry-references-model)
    - [8.5 Run V calibration](#85-run-v-calibration)
  - [9. Calibrate B and Finalize B/V Magnitudes](#9-calibrate-b-and-finalize-bv-magnitudes)
  - [10. Filter and Calculate Corrected Photometry](#10-filter-and-calculate-corrected-photometry)
  - [11. Estimate Distance and Attempt Age Measurement with a ZAMS Overlay](#11-estimate-distance-and-attempt-age-measurement-with-a-zams-overlay)
  - [12. R-Band Calibration and Color–Color Screening](#12-r-band-calibration-and-colorcolor-screening)
  - [13. Remove Photometric Non-Member Candidates and Regenerate the CMD](#13-remove-photometric-non-member-candidates-and-regenerate-the-cmd)
  - [14. Spectal Class](#14-spectal-class)
  - [15. Estimate Distance and Age with a ZAMS Overlay](#15-estimate-distance-and-age-with-a-zams-overlay)
- [16. Gaia Astrometric Membership Analysis](#16-gaia-astrometric-membership-analysis)
  - [16.1 — Query Gaia around the NGC 2477 field](#161-query-gaia-around-the-ngc-2477-field)
  - [16.2 — Cross-match the photometric stars to Gaia spatially](#162-cross-match-the-photometric-stars-to-gaia-spatially)
  - [16.3 — Determine the NGC 2477 astrometric concentration](#163-determine-the-ngc-2477-astrometric-concentration)
  - [16.4 — Assign membership to the photometric stars using sigma cuts](#164-assign-membership-to-the-photometric-stars-using-sigma-cuts)
  - [16.5 — Treat stars without Gaia matches as non-members](#165-treat-stars-without-gaia-matches-as-non-members)
  - [16.6 — Keep Gaia probable members only](#166-keep-gaia-probable-members-only)
  - [16.7 Estimate NGC 2477 Age and Distance Using Gaia Probable Members Only](#167-estimate-ngc-2477-age-and-distance-using-gaia-probable-members-only)
- [17. Gaia Membership After Color-Color Screening](#17-gaia-membership-after-color-color-screening)
  - [17.1 Load the post-screen photometric catalog](#171-load-the-post-screen-photometric-catalog)
  - [17.2 Convert screened stars to RA/Dec and cross-match with Gaia](#172-convert-screened-stars-to-radec-and-cross-match-with-gaia)
  - [17.3 Apply the same Gaia sigma-cut membership criteria](#173-apply-the-same-gaia-sigma-cut-membership-criteria)
  - [17.4 Keep color-screened + Gaia probable members only](#174-keep-color-screened-gaia-probable-members-only)
  - [17.5 Estimate distance and age using color-screened Gaia members](#175-estimate-distance-and-age-using-color-screened-gaia-members)
  - [17.6 CMD for color-screened Gaia probable members](#176-cmd-for-color-screened-gaia-probable-members)

# NGC 2477 Open Cluster Age and Distance

This is a NASA SEES summer internship project. The goal is to determine age and distance of NGC2477 open cluster


**Pixel-coordinate convention.** All star positions in this notebook use zero-based `(x, y)` pixel coordinates in the NumPy/Photutils arrays of their respective FITS images (`x` increases rightward, `y` increases with array row index). The reference-star positions and inter-image alignment shifts below are specified directly in those coordinates. The original V, B and R stacks are expected to have matching dimensions (2048 × 2048 for the data used to develop this notebook).

In [ ]:
# Global definitions
PROJECT_ROOT = "/data/projects/isef_open_cluster"
B_STACK = PROJECT_ROOT + "/data/NGC_2477/NGC_2477_b_stacked.fits"
V_STACK = PROJECT_ROOT + "/data/NGC_2477/NGC_2477_v_stacked.fits"
R_STACK = PROJECT_ROOT + "/data/NGC_2477/NGC_2477_r_stacked.fits"
OUTPUT = PROJECT_ROOT + "/output/NGC2477.csv"
RAW_OUTPUT = PROJECT_ROOT + "/output/NGC2477_raw.csv"
QC_OUTPUT = PROJECT_ROOT + "/output/NGC2477_quality_flags.csv"
OUTPUT_FILTERED = PROJECT_ROOT + "/output/NGC2477_filtered.csv"
OUTPUT_CALCED = PROJECT_ROOT + "/output/NGC2477_calculated.csv"
NOT_BELONG = PROJECT_ROOT + "/output/NGC2477_not_belong.csv"
CMD_PLOT = PROJECT_ROOT + "/output/NGC2477_CMD.png"

# Aperture settings in pixels
APERTURE_RADIUS = 6
SKY_INNER = 10
SKY_OUTER = 18

# Detection / filtering parameters
DETECTION_SIGMA = 5.0
FWHM_INIT = 5.0
EDGE_MARGIN = SKY_OUTER + 3
# Saturation cutoff in native stacked-image units (None disables this check).
# Set only after confirming the detector/stacking saturation level.
SATURATION_LEVEL = None

# Safety filters
FWHM_OVERLAP_FACTOR = 2.0
# Stricter check against ALL raw DAO detections.
# This removes stars whose aperture/sky annulus would be contaminated by nearby stars.
RAW_CONTAMINATION_DISTANCE = 20  # pixels; test sensitivity below
MIN_SNR = 5.0
MIN_FWHM = 1.5
MAX_FWHM = 20.0
MIN_PEAK_ABOVE_BG = 200

CROWDING_DISTANCE = 20


## 1. Import libraries and Setup Environement

This notebook uses `astropy`, `photutils`, `scipy`, `pandas`, `plotly`, `nbformat`, and `matplotlib`.

If needed:

```bash
pip install astropy photutils scipy pandas matplotlib
```


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from scipy.spatial import cKDTree
import warnings


## 2. FITS loading and reusable star detector

`detect_candidate_stars()` reads a stacked FITS image, estimates its sky background with sigma-clipped statistics, and detects star-like peaks using `DAOStarFinder`. The input is an **image path**; the output is a **pandas DataFrame** containing all candidate detections (including pixel coordinates and measured detection properties).

The function does **not** reject crowded or low-quality stars. Local FWHM and image-quality screening are handled separately in Sections 4–6. Background statistics are attached as DataFrame metadata (`.attrs`) for use by subsequent quality checks.


In [ ]:
def detect_candidate_stars(image_path, fwhm_init=5.0, detection_sigma=5.0,
                           background_clip_sigma=3.0):
    """Read one FITS image and return all DAOStarFinder detections as a DataFrame.

    Background median and standard deviation are stored in the DataFrame attrs
    for downstream image-quality checks. No QC filtering is performed here.
    """
    
    with fits.open(image_path) as hdul:
        image = hdul[0].data.astype(float)
        header = hdul[0].header.copy()

    if image.ndim != 2:
        raise ValueError(f"Expected a 2D FITS image: {image_path}")

    mean, median, std = sigma_clipped_stats(image, sigma=background_clip_sigma)
    if not np.isfinite(std) or std <= 0:
        raise ValueError(f"Invalid background noise estimate for {image_path}: {std}")

    finder = DAOStarFinder(
        fwhm=fwhm_init,
        threshold=detection_sigma * std,
    )
    sources = finder(image - median)
    if sources is None or len(sources) == 0:
        raise RuntimeError(
            f"No stars detected in {image_path}. Adjust detection_sigma or fwhm_init."
        )

    detected = sources.to_pandas().rename(columns={
        "xcentroid": "x_centroid",
        "ycentroid": "y_centroid",
        "mag": "magnitude",
    })
    detected.attrs.update({
        "background_mean": float(mean),
        "background_median": float(median),
        "background_std": float(std),
    })
    return image, header, detected


## 3. Detect candidate stars

Run the reusable detector on the V-band stacked image, which serves as the reference for subsequent B/V/R photometry. **The 5σ detection threshold** (`DETECTION_SIGMA = 5.0`) is measured relative to the sigma-clipped sky-background standard deviation. Detections are not yet rejected for crowding, FWHM, edge proximity, or signal strength; these checks follow in Sections 4–6.

The FITS pixels are also kept in `data` for later FWHM measurement, QC, and aperture photometry.

### `DAOStarFinder` Parameters

`DAOStarFinder` (https://photutils.readthedocs.io/en/stable/api/photutils.detection.DAOStarFinder.html) detects star-like brightness peaks using an expected stellar profile.

| Parameter | Description |
|-----------|-------------|
| `threshold` | Minimum detection level above the background, expressed in image units; here `detection_sigma × background_std`. |
| `fwhm` | Expected stellar Full Width at Half Maximum in pixels, used for the detection kernel. |
| `sharplo` / `sharphi` | Bounds on source sharpness. |
| `roundlo` / `roundhi` | Bounds on source roundness. |
| `peakmax` | Optional upper bound on peak value. |
| `exclude_border` | Whether to exclude detections near the image edge. |
| `brightest` | Optional limit on how many of the brightest detections to return. |

The main detection settings in this notebook are `DETECTION_SIGMA` and `FWHM_INIT`; they can be adjusted for other images.


In [ ]:
# Detect stars with the reusable image-path -> DataFrame function.
data, header, df = detect_candidate_stars(
    V_STACK,
    fwhm_init=FWHM_INIT,
    detection_sigma=DETECTION_SIGMA,
)

ny, nx = data.shape
mean = df.attrs["background_mean"]
median = df.attrs["background_median"]
std = df.attrs["background_std"]

print(f"Loaded {V_STACK}: {nx} x {ny}")
print("Filter:", header.get("FILTER", "Unknown"))
print(f"Background median = {median:.2f}, std = {std:.2f}")
print(f"Detected candidate stars: {len(df)}")

df.to_csv(RAW_OUTPUT, index=False)
df.head()


## 4. Estimate FWHM for each detected star

A simple local half-maximum method is used to reject clearly blended or abnormal detections. Although stars are point sources in space, they appear as small blurred spots rather than single pixels in CCD images. The size of this spot is commonly described by its Full Width at Half Maximum (FWHM), which measures the width of the star's brightness profile. By estimating the FWHM for every detected star, we can identify detections that are significantly larger or smaller than typical stars. These unusual detections are more likely to be blended stars, cosmic rays, or other unreliable measurements and can be filtered before performing aperture photometry.
### Full Width at Half Maximum (FWHM)

**Full Width at Half Maximum (FWHM)** measures the apparent width of a star's brightness profile at half its maximum intensity, after subtracting the background.

- **Small FWHM:** A sharp, well-focused star.
- **Large FWHM:** A blurry star or potentially overlapping stars.

FWHM is usually measured in pixels.

#### How FWHM Is Calculated

The function `estimate_fwhm_local()`:

1. Extracts a 15×15 pixel region around the star.
2. Estimates the local background using the median pixel intensity.
3. Finds the star's peak brightness above the background.
4. Counts the pixels brighter than half that peak (above background).
5. Converts that area into the diameter of an equivalent circle:

$$
\mathrm{FWHM} \approx 2\sqrt{\frac{\mathrm{Area}}{\pi}}
$$

`DAOStarFinder` uses the expected FWHM to recognize star-like objects. In this notebook, we also estimate each detected star's FWHM and reject stars with unusually large values, since they are more likely to be blended with nearby stars.


In [ ]:
def estimate_fwhm_local(image, x, y, box=15):
    # Estimate FWHM from a small image cutout using area above half maximum.
    x0, y0 = int(round(x)), int(round(y))
    half = box // 2

    x1, x2 = max(0, x0 - half), min(image.shape[1], x0 + half + 1)
    y1, y2 = max(0, y0 - half), min(image.shape[0], y0 + half + 1)
    cutout = image[y1:y2, x1:x2]

    if cutout.size == 0:
        return np.nan

    local_median = np.median(cutout)
    peak = np.max(cutout)
    height = peak - local_median

    if height <= 0:
        return np.nan

    half_max = local_median + 0.5 * height
    area = np.sum(cutout >= half_max)

    # Approximate FWHM as diameter of a circle with the same half-max area.
    return 2.0 * np.sqrt(area / np.pi)


In [ ]:
df["fwhm_px"] = [
    estimate_fwhm_local(data, x, y)
    for x, y in zip(df["x_centroid"], df["y_centroid"])
]

median_fwhm = np.nanmedian(df["fwhm_px"])
print(f"Median estimated FWHM = {median_fwhm:.2f} px")

## 5. Add quality measurements

We compute nearest-neighbor distance, edge distance, peak value, simple SNR estimate, and relative magnitude.

In crowded star fields, nearby stars can overlap within the photometry aperture, causing their light to blend together and producing inaccurate brightness measurements. To identify these cases efficiently, a KD-tree (k-dimensional tree) is built from the detected star coordinates, allowing the nearest neighboring star for each detection to be found much faster than comparing every pair of stars. The nearest-neighbor distance is then compared with a separation threshold chosen to protect the aperture and sky annulus, and stars with neighbors that are too close are excluded because their photometry is more likely to be contaminated by overlapping light. This filtering step improves the overall reliability of the measured magnitudes used in the subsequent color–magnitude diagram (CMD).


`apply_star_qc()` combines these measurements with the Section 6 filtering checks. Neighbor distances are calculated against **all original detections**, before any stars are rejected, so a problematic neighbor is not overlooked. The SNR is an approximate value derived from detection flux and global background noise.


In [ ]:
def apply_star_qc(stars, image, background_median, background_std, qc_params):
    """Measure quality and add QC flags for every detected star (no row removal).

    stars must contain x_centroid, y_centroid, fwhm_px, flux, and n_pixels.
    qc_params holds the filter thresholds; saturation_level=None disables that check.
    """
    required = {"x_centroid", "y_centroid", "fwhm_px", "flux", "n_pixels"}
    missing = required - set(stars.columns)
    if missing:
        raise ValueError(f"Missing star columns: {sorted(missing)}")
    if stars.empty:
        raise ValueError("The star catalog is empty.")
    if image.ndim != 2 or not np.isfinite(background_std) or background_std <= 0:
        raise ValueError("Need a 2D image and a positive background standard deviation.")

    result = stars.copy()
    x = result["x_centroid"].to_numpy(dtype=float)
    y = result["y_centroid"].to_numpy(dtype=float)
    if not np.all(np.isfinite(x) & np.isfinite(y)):
        raise ValueError("Detected star coordinates must be finite.")

    # Nearest neighbor among ALL candidate detections (before filtering).
    positions = np.column_stack((x, y))
    if len(result) >= 2:
        distances, _ = cKDTree(positions).query(positions, k=2)
        result["nearest_neighbor_px"] = distances[:, 1]
    else:
        result["nearest_neighbor_px"] = np.inf

    ny, nx = image.shape
    result["edge_distance_px"] = np.minimum.reduce([
        x, y, nx - x, ny - y
    ])

    # Highest pixel value in a 5x5 patch centered on each detection.
    def local_peak(xpos, ypos, box=5):
        xc, yc = int(round(xpos)), int(round(ypos))
        half = box // 2
        patch = image[
            max(0, yc - half):min(ny, yc + half + 1),
            max(0, xc - half):min(nx, xc + half + 1),
        ]
        return np.max(patch) if patch.size else np.nan

    result["peak"] = [local_peak(xi, yi) for xi, yi in zip(x, y)]
    result["peak_above_bg"] = result["peak"] - background_median
    result["snr"] = (
        result["flux"]
        / (background_std * np.sqrt(np.maximum(result["n_pixels"], 1)))
    )
    result["relative_mag"] = -2.5 * np.log10(
        np.maximum(result["flux"], 1e-12)
    )

    # Image-quality checks. These do NOT classify cluster membership.
    result["pass_fwhm_overlap"] = (
        result["nearest_neighbor_px"]
        >= qc_params["fwhm_overlap_factor"] * result["fwhm_px"]
    )
    result["pass_photometry_safety"] = (
        result["nearest_neighbor_px"] >= qc_params["crowding_distance"]
    )
    result["pass_edge"] = (
        result["edge_distance_px"] >= qc_params["edge_margin"]
    )
    result["pass_snr"] = result["snr"] >= qc_params["min_snr"]
    result["pass_peak_strength"] = (
        result["peak_above_bg"] >= qc_params["min_peak_above_bg"]
    )
    result["pass_fwhm_valid"] = (
        np.isfinite(result["fwhm_px"])
        & (result["fwhm_px"] > qc_params["min_fwhm"])
        & (result["fwhm_px"] < qc_params["max_fwhm"])
    )

    filter_cols = [
        "pass_fwhm_overlap", "pass_photometry_safety", "pass_edge",
        "pass_snr", "pass_peak_strength", "pass_fwhm_valid",
    ]
    saturation_level = qc_params.get("saturation_level")
    if saturation_level is not None:
        result["pass_saturation"] = result["peak"] < saturation_level
        filter_cols.append("pass_saturation")

    result["qc_pass"] = result[filter_cols].all(axis=1)
    result["qc_reasons"] = result[filter_cols].apply(
        lambda row: ", ".join(
            name.removeprefix("pass_") for name, passed in row.items()
            if not passed
        ),
        axis=1,
    )
    return result


## 6. Apply filters

Key separation filters:

1. **FWHM overlap filter**: reject stars where the nearest neighbor is closer than `2 × FWHM`.
2. **Crowding filter**: Stars with a nearby neighbor that could overlap the APERTURE_RADIUS (6 px) aperture or the SKY_INNER–SKY_OUTER (10–18 px) sky annulus are excluded to reduce contamination from overlapping light.

3. **Other quality checks:** enforce image-edge clearance, minimum approximate SNR and peak strength, and a plausible FWHM range. Saturation is tested only if an appropriate stacked-image threshold has been established.

These are **image-quality checks**, not color–color screening (CCS) or Gaia membership selection. Every detection and its rejection reasons are preserved in `QC_OUTPUT`; the retained subset is used by subsequent photometry. A crowding-threshold sensitivity check compares alternate cutoffs without changing the adopted value.


In [ ]:
def applyFilters(df, dataFileWithQC):
    # Settings for the reusable measurement-and-filtering function.
    qc_params = {
        "fwhm_overlap_factor": FWHM_OVERLAP_FACTOR,
        "crowding_distance": RAW_CONTAMINATION_DISTANCE,
        "edge_margin": EDGE_MARGIN,
        "min_snr": MIN_SNR,
        "min_peak_above_bg": MIN_PEAK_ABOVE_BG,
        "min_fwhm": MIN_FWHM,
        "max_fwhm": MAX_FWHM,
        "saturation_level": SATURATION_LEVEL,
    }

    # One call computes all quality measurements and flags; no stars are discarded.
    if os.path.exists(dataFileWithQC):
        df = pd.read_csv(dataFileWithQC)
        print(f"Loaded full QC table ({len(df)} detected stars) from {dataFileWithQC}")
    else:
        df = apply_star_qc(df, data, median, std, qc_params)
        df.to_csv(dataFileWithQC, index=False)
        print(f"Saved full QC table ({len(df)} detected stars) to {dataFileWithQC}")

    filter_cols = [col for col in df.columns if col.startswith("pass_")]
    print("\nFilter summary:")
    for col in filter_cols:
        print(f"{col:24s}: {df[col].sum():5d} / {len(df)}")
    if SATURATION_LEVEL is None:
        print("Saturation check disabled (no validated threshold for the stacked image).")

    clean = df.loc[df["qc_pass"]].copy()
    print(f"\nClean stars after all filters: {len(clean)}")

    # Sensitivity of the retained sample to the crowding cutoff only.
    other_filters = [c for c in filter_cols if c != "pass_photometry_safety"]
    print("\nCrowding distance sensitivity (other QC criteria held fixed):")
    for distance in sorted({18, RAW_CONTAMINATION_DISTANCE, 24}):
        mask = (
            df[other_filters].all(axis=1)
            & (df["nearest_neighbor_px"] >= distance)
        )
        print(f"{distance:2d} px: {mask.sum():5d} stars retained")

    return clean


In [ ]:
cleanDf = applyFilters(df, QC_OUTPUT)

## 7. Save the quality-controlled star catalog

Retain **all stars that pass the Section 5–6 image-quality checks**, without any additional brightness sampling. Assign a sequential `StarID` while preserving the original DAO `id` for traceability to the full QC table. Export the accepted stars to CSV for subsequent photometry and analysis.

In [ ]:
# Retain every QC-passing star and assign a stable catalog number.
selected = cleanDf.copy().reset_index(drop=True)
selected["StarID"] = np.arange(1, len(selected) + 1)

output_cols = [
    "StarID",
    "id",  # original DAO detection ID for joining to QC_OUTPUT
    "x_centroid",
    "y_centroid",
    "relative_mag",
    "flux",
    "peak",
    "peak_above_bg",
    "snr",
    "fwhm_px",
    "nearest_neighbor_px",
    "edge_distance_px",
]

selected[output_cols].to_csv(OUTPUT, index=False)
print(f"Saved {len(selected)} stars to {OUTPUT}")


## 8. Automated V-Band Calibration Using Gaia DR3 Synthetic Photometry

### Scientific idea

Instrumental magnitude is calculated from background-subtracted aperture flux as
`m_inst = -2.5 log10(flux)`. A reference star provides the difference between
its catalog magnitude and instrumental magnitude (a **photometric zero point**).
Multiple catalog-matched reference stars yield a robust zero point; a fitted
variation of that difference with catalog B−V tests for a **filter color term**.

**Reference catalog:** Gaia DR3 `synthetic_photometry_gspc` provides synthetic
Johnson–Kron–Cousins B, V and R magnitudes, joined with Gaia source positions.
These are used **only to calibrate photometry**, not to decide cluster membership.
The same helpers will calibrate B in Section 9 and R in Section 12.

Documentation: [Gaia DR3 GSPC](https://gaia.aip.de/metadata/gaiadr3/synthetic_photometry_gspc/) ·
[ESA Gaia ADQL guide](https://www.cosmos.esa.int/web/gaia-users/archive/writing-queries)

### 8.1 Obtain a WCS solution (pixel position → RA/Dec)

A **World Coordinate System (WCS)** maps pixel centroids to positions on the sky.
Use the FITS WCS header if present; otherwise reuse a saved plate solution or
ask Astrometry.net to solve the V image. This single V-band solution is also
reused by the later Gaia astrometric-membership analysis.

Astrometry.net authenticates with a private API key and returns a WCS FITS
header. Set the `ASTROMETRY_NET_API_KEY` environment variable; the notebook
prompts for the key only when a new solution is required. No key is saved.

In [ ]:
from pathlib import Path
from getpass import getpass
from astropy.wcs import WCS

v_stack_path = Path(V_STACK)
with fits.open(v_stack_path) as hdul:
    original_v_header = hdul[0].header.copy()
    original_v_shape = hdul[0].data.shape

original_v_wcs = WCS(original_v_header)
V_STACK_HAS_WCS = bool(original_v_wcs.has_celestial)
print("V image celestial WCS in header:", V_STACK_HAS_WCS)

In [ ]:
import json
import time
import requests
import os
from pathlib import Path

ASTROMETRY_API_BASE = "https://nova.astrometry.net"
AstrometryAPIKey = os.getenv("ASTROMETRY_NET_API_KEY") or os.getenv("AstrometryAPIKey")
ASTROMETRY_POLL_SECONDS = 5
ASTROMETRY_MAX_POLLS = 180

def _astrometry_post(endpoint, request_json, files=None):
    url = f"{ASTROMETRY_API_BASE}{endpoint}"
    data = {"request-json": json.dumps(request_json)}
    response = requests.post(url, data=data, files=files, timeout=120)
    response.raise_for_status()
    result = response.json()
    if result.get("status") == "error":
        raise RuntimeError(f"Astrometry.net API error: {result}")
    return result

def astrometry_login(api_key):
    if not api_key:
        raise ValueError("Set ASTROMETRY_NET_API_KEY in the environment to enable plate solving.")
    result = _astrometry_post("/api/login", {"apikey": api_key})
    session = result.get("session")
    if not session:
        raise RuntimeError(f"Astrometry.net login did not return a session: {result}")
    return session

def astrometry_upload(fits_path, session):
    fits_path = Path(fits_path)
    with open(fits_path, "rb") as f:
        files = {"file": (fits_path.name, f, "application/fits")}
        result = _astrometry_post(
            "/api/upload",
            {
                "session": session,
                "publicly_visible": "n",
                "allow_modifications": "n",
                "allow_commercial_use": "n",
            },
            files=files,
        )
    subid = result.get("subid")
    if subid is None:
        raise RuntimeError(f"Astrometry.net upload did not return subid: {result}")
    return subid

def astrometry_wait_for_job(subid, max_polls=ASTROMETRY_MAX_POLLS):
    url = f"{ASTROMETRY_API_BASE}/api/submissions/{subid}"
    for _ in range(max_polls):
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        result = response.json()
        jobs = [j for j in result.get("jobs", []) if j is not None]
        if jobs:
            return jobs[0]
        time.sleep(ASTROMETRY_POLL_SECONDS)
    raise TimeoutError(f"No Astrometry.net job created for submission {subid}.")

def astrometry_wait_for_success(job_id, max_polls=ASTROMETRY_MAX_POLLS):
    url = f"{ASTROMETRY_API_BASE}/api/jobs/{job_id}"
    for _ in range(max_polls):
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        result = response.json()
        status = result.get("status")
        if status == "success":
            return result
        if status == "failure":
            raise RuntimeError(f"Astrometry.net plate solve failed for job {job_id}.")
        time.sleep(ASTROMETRY_POLL_SECONDS)
    raise TimeoutError(f"Astrometry.net job {job_id} did not finish in time.")

def astrometry_download_wcs(job_id, output_path):
    output_path = Path(output_path)
    url = f"{ASTROMETRY_API_BASE}/wcs_file/{job_id}"
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    output_path.write_bytes(response.content)
    return output_path

In [ ]:
# Retain compatibility with the earlier cached NGC 2477 plate solution.
_wcs_default = Path(V_STACK).with_name(Path(V_STACK).stem + "_astrometry_wcs.fits")
_wcs_legacy = Path(V_STACK).parent / "NGC2477_v_astrometry_wcs.fits"
ASTROMETRY_WCS_FILE = (
    _wcs_legacy if ("NGC_2477" in Path(V_STACK).name and _wcs_legacy.exists())
    else _wcs_default
)

if V_STACK_HAS_WCS:
    astrometric_wcs = original_v_wcs
    ASTROMETRY_JOB_ID = None
elif ASTROMETRY_WCS_FILE.exists():
    with fits.open(ASTROMETRY_WCS_FILE) as hdul:
        astrometric_wcs = WCS(hdul[0].header)
    ASTROMETRY_JOB_ID = None
    print("Reusing cached WCS:", ASTROMETRY_WCS_FILE)
else:
    if not AstrometryAPIKey:
        AstrometryAPIKey = getpass("Astrometry.net API key (hidden): ").strip()
    session = astrometry_login(AstrometryAPIKey)
    submission_id = astrometry_upload(V_STACK, session)
    ASTROMETRY_JOB_ID = astrometry_wait_for_job(submission_id)
    astrometry_wait_for_success(ASTROMETRY_JOB_ID)
    astrometry_download_wcs(ASTROMETRY_JOB_ID, ASTROMETRY_WCS_FILE)
    with fits.open(ASTROMETRY_WCS_FILE) as hdul:
        astrometric_wcs = WCS(hdul[0].header)
    print("Saved WCS:", ASTROMETRY_WCS_FILE)

if not astrometric_wcs.has_celestial:
    raise RuntimeError("A usable celestial WCS could not be obtained for the V image.")
print("WCS ready for calibration and later Gaia membership analysis.")

### 8.2 Query Gaia DR3 synthetic B/V/R reference magnitudes

`astroquery.gaia.Gaia.launch_job_async()` submits an **ADQL** (Astronomical
Data Query Language) query to the Gaia archive. The query joins
`gaiadr3.gaia_source` (positions, Gaia quality indicators) with
`gaiadr3.synthetic_photometry_gspc` (synthetic B/V/R magnitudes and flags)
on `source_id`. The search footprint is computed from the WCS image corners;
no cluster coordinates need to be hardcoded.

Results are cached as CSV to avoid repeating a large remote query. A Gaia
synthetic-photometry source is a candidate **calibration reference**, not
necessarily a cluster member. `*_jkc_flag == 1` indicates magnitudes within
the catalog's validated magnitude/color range.

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.gaia import Gaia

CALIBRATION_OUTPUT_DIR = Path(PROJECT_ROOT) / "output"
CALIBRATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CALIBRATION_PREFIX = Path(V_STACK).stem
GAIA_GSPC_CACHE = CALIBRATION_OUTPUT_DIR / f"{CALIBRATION_PREFIX}_gaia_gspc.csv"
GAIA_CALIBRATION_MATCH_RADIUS_ARCSEC = 1.5
MIN_CALIBRATION_REFERENCES = 6


def query_gaia_synthetic_photometry(wcs, shape, cache_path):
    """Query Gaia DR3 GSPC for the V-image footprint, or load a cached result."""
    cache_path = Path(cache_path)
    if cache_path.exists():
        catalog = pd.read_csv(cache_path, dtype={"source_id": "string"})
        print(f"Loaded {len(catalog):,} Gaia synthetic references from {cache_path}")
        return catalog

    ny, nx = shape
    corners = wcs.pixel_to_world(
        np.array([0, nx - 1, nx - 1, 0], dtype=float),
        np.array([0, 0, ny - 1, ny - 1], dtype=float),
    )
    mean_direction = corners.cartesian.xyz.value.mean(axis=1)
    mean_direction /= np.linalg.norm(mean_direction)
    center = SkyCoord(x=mean_direction[0], y=mean_direction[1],
                      z=mean_direction[2], representation_type="cartesian", frame="icrs")
    radius_deg = float((1.05 * center.separation(corners).max()).to_value(u.deg))
    ra_deg = float(center.spherical.lon.deg)
    dec_deg = float(center.spherical.lat.deg)

    adql = f"""
    SELECT g.source_id, g.ra, g.dec, g.ruwe, g.phot_variable_flag,
           s.b_jkc_mag, s.v_jkc_mag, s.r_jkc_mag,
           s.b_jkc_flag, s.v_jkc_flag, s.r_jkc_flag
    FROM gaiadr3.gaia_source AS g
    JOIN gaiadr3.synthetic_photometry_gspc AS s ON g.source_id = s.source_id
    WHERE 1 = CONTAINS(
        POINT('ICRS', g.ra, g.dec),
        CIRCLE('ICRS', {ra_deg:.10f}, {dec_deg:.10f}, {radius_deg:.10f})
    )
    """
    result = Gaia.launch_job_async(adql).get_results().to_pandas()
    if result.empty:
        raise RuntimeError("No Gaia DR3 synthetic photometry returned for the image.")
    result["source_id"] = result["source_id"].astype("string")
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(cache_path, index=False)
    print(f"Gaia GSPC: {len(result):,} field sources; cached at {cache_path}")
    return result


gaia_synthetic = query_gaia_synthetic_photometry(
    astrometric_wcs, data.shape, GAIA_GSPC_CACHE
)

### 8.3 Match detected stars to Gaia (calibration only)

Use `WCS.pixel_to_world()` to obtain RA/Dec from V-image positions.
`SkyCoord.match_to_catalog_sky()` finds the nearest Gaia source on the
celestial sphere. Matches must be within **1.5 arcsec**, unambiguous, and
one-to-one. The catalog's Gaia ID and synthetic magnitudes are attached only
to the **calibration-candidate table**; later Gaia membership selection runs
independently, with its own complete Gaia astrometric field catalog.

In [ ]:
def match_gaia_calibration_stars(stars, wcs, catalog, radius_arcsec=1.5):
    """Attach unique, close Gaia GSPC matches to the V-position star catalog."""
    if len(catalog) < 2:
        raise RuntimeError("At least two Gaia GSPC sources are needed for ambiguity checks.")
    candidates = stars[["StarID", "V_x_centroid", "V_y_centroid"]].copy().reset_index(drop=True)
    star_sky = wcs.pixel_to_world(
        candidates["V_x_centroid"].to_numpy(dtype=float),
        candidates["V_y_centroid"].to_numpy(dtype=float),
    )
    cat_sky = SkyCoord(ra=catalog["ra"].to_numpy(dtype=float)*u.deg,
                       dec=catalog["dec"].to_numpy(dtype=float)*u.deg)
    nearest, separation, _ = star_sky.match_to_catalog_sky(cat_sky)
    _, second_sep, _ = star_sky.match_to_catalog_sky(cat_sky, nthneighbor=2)
    acceptable = ((separation <= radius_arcsec*u.arcsec)
                  & (second_sep > radius_arcsec*u.arcsec))

    candidates["GaiaMatchSepArcsec"] = separation.arcsec
    candidates["GaiaMatchValid"] = acceptable
    candidates["Gaia_source_id"] = pd.Series(pd.NA, index=candidates.index, dtype="string")
    cols = ["ruwe", "phot_variable_flag", "b_jkc_mag", "v_jkc_mag", "r_jkc_mag",
            "b_jkc_flag", "v_jkc_flag", "r_jkc_flag"]
    for col in cols:
        candidates[f"Gaia_{col}"] = np.nan if col != "phot_variable_flag" else None

    ids = np.flatnonzero(acceptable)
    if len(ids):
        selected_rows = catalog.iloc[nearest[ids]]
        candidates.loc[ids, "Gaia_source_id"] = selected_rows["source_id"].astype("string").to_numpy()
        for col in cols:
            candidates.loc[ids, f"Gaia_{col}"] = selected_rows[col].to_numpy()

    # Only the closest of repeated matches to any one Gaia source can be accepted.
    dup = candidates["GaiaMatchValid"] & candidates["Gaia_source_id"].duplicated(keep=False)
    for _, group in candidates.loc[dup].groupby("Gaia_source_id"):
        discard = group.sort_values("GaiaMatchSepArcsec").index[1:]
        candidates.loc[discard, "GaiaMatchValid"] = False

    print(f"Unique Gaia calibration matches: {int(candidates['GaiaMatchValid'].sum()):,} / {len(candidates):,}")
    return candidates

# Keep the original StarID and native zero-based V pixel coordinates.
selected = selected.rename(columns={"x_centroid": "V_x_centroid",
                                    "y_centroid": "V_y_centroid"})
calibration_matches = match_gaia_calibration_stars(
    selected, astrometric_wcs, gaia_synthetic,
    radius_arcsec=GAIA_CALIBRATION_MATCH_RADIUS_ARCSEC,
)

### 8.4 Reusable calibration helpers: photometry → references → model

The reusable `calibrate_band()` function delegates the scientific operations
to small helpers:

1. `aperture_flux_at_positions()`: measure the source flux in a circular aperture;
   estimate local sky flux with an annulus and subtract it.
2. `measure_band_photometry()`: measure all eligible stars in V, B or R using the
   same aperture and record instrumental magnitudes.
3. `select_calibration_references()`: retain close Gaia matches with validated
   band magnitudes, no Gaia variability flag, reasonable RUWE and positive flux.
4. `fit_color_calibration()`: robustly fit **catalog magnitude − instrumental
   magnitude** versus catalog B−V, remove anomalous references using MAD, and
   test whether a nonzero color slope is supported.
5. `calibrate_band()`: fit the zero point and optional color term for one band.
6. `calibrate_photometry(image_path, stars, band, ...)`: reusable top-level call;
   reads the FITS image, registers B/R to V, and returns calibrated magnitudes,
   the solution, the reference audit, and registration diagnostics.

A simple zero point is used unless the color trend is significant and the
reference stars span enough color range. B and V are finalized together once
both bands are measured, so their color corrections can use the science star's
**measured B−V**, not its Gaia magnitude. This works for stars absent from GSPC.

When enough references are available, a reserved subset is **held out** to check the transformation on stars not used for fitting. Each band also saves a diagnostic plot of catalog-minus-instrumental magnitude against Gaia B−V. A color term is not adopted if its validation performance is clearly worse than a zero point alone.

Dependency: `astroalign` is needed for B/R registration (`%pip install astroalign` in Jupyter if not installed).

The calibrated magnitude is computed by adding the zero point (and, when validated, the color term) to the instrumental magnitude. Testing catalog-minus-instrumental magnitude versus Gaia B−V is an **empirical passband check**, not a laboratory measurement of filter transmission.


In [ ]:
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from scipy.optimize import least_squares
from scipy.stats import linregress
try:
    import astroalign
except ImportError as exc:
    raise ImportError("This notebook needs astroalign. Install it with: %pip install astroalign") from exc
import json


def aperture_flux_at_positions(image, x, y, r, sky_in, sky_out):
    """Sky-subtracted aperture flux; NaN for out-of-bounds annuli."""
    x = np.atleast_1d(np.asarray(x, dtype=float))
    y = np.atleast_1d(np.asarray(y, dtype=float))
    if x.shape != y.shape:
        raise ValueError("x and y must have identical shapes")
    ny, nx = image.shape
    valid = (np.isfinite(x) & np.isfinite(y)
             & (x >= sky_out) & (x < nx - sky_out)
             & (y >= sky_out) & (y < ny - sky_out))
    flux = np.full(len(x), np.nan)
    if valid.any():
        positions = np.column_stack([x[valid], y[valid]])
        aperture = CircularAperture(positions, r=r)
        annulus = CircularAnnulus(positions, r_in=sky_in, r_out=sky_out)
        aperture_sum = np.asarray(aperture_photometry(image, aperture)["aperture_sum"], dtype=float)
        annulus_sum = np.asarray(aperture_photometry(image, annulus)["aperture_sum"], dtype=float)
        flux[valid] = aperture_sum - (annulus_sum / annulus.area) * aperture.area
    return flux


def estimate_band_registration(band_image, v_image, max_rms_px=2.0):
    """Return a B/R -> V transform (no image resampling for photometry)."""
    transform, (band_points, v_points) = astroalign.find_transform(band_image, v_image)
    fitted_v = transform(band_points)
    residuals = np.linalg.norm(fitted_v - v_points, axis=1)
    rms = float(np.sqrt(np.mean(residuals ** 2)))
    if len(residuals) < 5 or rms > max_rms_px:
        raise RuntimeError(f"Registration unreliable: {len(residuals)} matches; RMS={rms:.2f}px")
    return transform, {"n_matches": len(residuals), "rms_px": rms,
                       "rotation_deg": float(np.degrees(transform.rotation)),
                       "scale": float(transform.scale)}


def measure_band_photometry(stars, image, band, v_to_band=None,
                            aperture_radius=6, sky_in=10, sky_out=18):
    """Measure a band on the original image, mapping V positions when required."""
    band = band.upper()
    result = stars.copy()
    v_positions = result[["V_x_centroid", "V_y_centroid"]].to_numpy(dtype=float)
    band_positions = v_positions if v_to_band is None else v_to_band.inverse(v_positions)
    result[f"{band}_x_centroid"] = band_positions[:, 0]
    result[f"{band}_y_centroid"] = band_positions[:, 1]
    ny, nx = image.shape
    x = band_positions[:, 0]
    y = band_positions[:, 1]
    result[f"{band}_position_valid"] = (
        np.isfinite(x) & np.isfinite(y)
        & (x >= sky_out) & (x < nx - sky_out)
        & (y >= sky_out) & (y < ny - sky_out)
    )
    result[f"{band}_flux"] = aperture_flux_at_positions(
        image, x, y, aperture_radius, sky_in, sky_out
    )
    flux = result[f"{band}_flux"].to_numpy(dtype=float)
    inst = np.full(len(result), np.nan)
    valid = np.isfinite(flux) & (flux > 0)
    inst[valid] = -2.5 * np.log10(flux[valid])
    result[f"{band}_inst_mag"] = inst
    return result


def select_calibration_references(stars, matched, band):
    """Use Gaia validation and basic source-quality criteria, not membership."""
    col = f"{band.lower()}_jkc_mag"
    flag = f"{band.lower()}_jkc_flag"
    refs = stars[["StarID", f"{band}_inst_mag", f"{band}_flux",
                  f"{band}_position_valid"]].merge(matched, on="StarID", validate="one_to_one")
    refs["Gaia_BV"] = pd.to_numeric(refs["Gaia_b_jkc_mag"], errors="coerce") - pd.to_numeric(
        refs["Gaia_v_jkc_mag"], errors="coerce")
    catalog_mag = pd.to_numeric(refs[f"Gaia_{col}"], errors="coerce")
    ruwe = pd.to_numeric(refs["Gaia_ruwe"], errors="coerce")
    ref_ok = (
        refs["GaiaMatchValid"].fillna(False).astype(bool)
        & np.isfinite(catalog_mag) & np.isfinite(refs["Gaia_BV"])
        & (pd.to_numeric(refs[f"Gaia_{flag}"], errors="coerce") == 1)
        & (pd.to_numeric(refs["Gaia_b_jkc_flag"], errors="coerce") == 1)
        & (pd.to_numeric(refs["Gaia_v_jkc_flag"], errors="coerce") == 1)
        & (np.isfinite(ruwe) & (ruwe < 1.4))
        & (~refs["Gaia_phot_variable_flag"].fillna("").astype(str).str.upper().eq("VARIABLE"))
        & refs[f"{band}_position_valid"].astype(bool)
        & (pd.to_numeric(refs[f"{band}_flux"], errors="coerce") > 0)
    )
    refs["catalog_mag"] = catalog_mag
    refs["catalog_BV"] = refs["Gaia_BV"]
    refs["zp_individual"] = refs["catalog_mag"] - refs[f"{band}_inst_mag"]
    refs["initial_reference_ok"] = ref_ok
    return refs


def robust_linear_fit(x, y, pivot=None):
    """Fit y = zero + slope*(x-pivot) with Huber-type least squares."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if pivot is None:
        pivot = float(np.median(x))
    offsets = x - pivot
    guess = np.array([float(np.median(y)), 0.0])
    fit = least_squares(lambda c: y - (c[0] + c[1] * offsets),
                        guess, loss="huber", f_scale=0.05)
    return float(fit.x[0]), float(fit.x[1]), float(pivot)


def fit_color_calibration(refs, min_references=6, holdout_fraction=0.20):
    """Robust zero-point + color-term fit, with an independent check when possible."""
    ok_indices = np.flatnonzero(refs["initial_reference_ok"].to_numpy(dtype=bool))
    if len(ok_indices) < min_references:
        raise RuntimeError(f"Only {len(ok_indices)} usable Gaia reference stars; need {min_references}. "
                           "Check GSPC availability, WCS, image quality, and aperture settings.")

    # Keep a reproducible test subset completely out of reference-star fitting.
    validation_indices = np.array([], dtype=int)
    fit_indices = ok_indices
    if len(ok_indices) >= max(12, min_references + 5):
        rng = np.random.default_rng(42)
        n_validation = max(2, int(np.ceil(holdout_fraction * len(ok_indices))))
        validation_indices = rng.choice(ok_indices, n_validation, replace=False)
        fit_indices = np.setdiff1d(ok_indices, validation_indices)
    x_all = refs["catalog_BV"].to_numpy(dtype=float)
    y_all = refs["zp_individual"].to_numpy(dtype=float)
    eligible_training = np.zeros(len(refs), dtype=bool)
    eligible_training[fit_indices] = True
    keep = eligible_training.copy()

    for _ in range(4):
        zero, slope, pivot = robust_linear_fit(x_all[keep], y_all[keep])
        resid = y_all - zero - slope * (x_all - pivot)
        med = float(np.median(resid[keep]))
        mad = float(1.4826 * np.median(np.abs(resid[keep] - med)))
        updated = eligible_training & (np.abs(resid - med) <= max(0.06, 3.5 * mad))
        if updated.sum() < min_references:
            raise RuntimeError("Too few consistent Gaia calibration references after robust rejection.")
        if np.array_equal(keep, updated):
            break
        keep = updated

    _, fitted_slope, pivot = robust_linear_fit(x_all[keep], y_all[keep])
    span = float(np.ptp(x_all[keep]))
    if int(keep.sum()) >= 8 and span >= 0.4:
        trend = linregress(x_all[keep], y_all[keep])
        significant = (np.isfinite(trend.stderr)
                       and abs(trend.slope) >= 3 * max(float(trend.stderr), 1e-10)
                       and abs(trend.slope) * span >= 0.02)
    else:
        significant = False

    constant_zero = float(np.median(y_all[keep]))
    slope = fitted_slope if significant else 0.0
    zero = float(np.median(y_all[keep] - slope * (x_all[keep] - pivot)))
    holdout_constant_mae = None
    holdout_color_mae = None
    if len(validation_indices):
        val_x, val_y = x_all[validation_indices], y_all[validation_indices]
        holdout_constant_mae = float(np.median(np.abs(val_y - constant_zero)))
        holdout_color_mae = float(np.median(np.abs(val_y - zero - slope * (val_x - pivot))))
        # If a statistically significant slope does not generalize, retain a zero point only.
        if significant and (holdout_color_mae > holdout_constant_mae * 1.2
                            and holdout_color_mae - holdout_constant_mae > 0.015):
            significant = False
            slope = 0.0
            zero = constant_zero

    prediction = zero + slope * (x_all[keep] - pivot)
    offsets = y_all[keep] - prediction
    robust_scatter = float(1.4826 * np.median(np.abs(offsets - np.median(offsets))))

    audit = refs.copy()
    audit["calibration_used"] = keep
    audit["calibration_role"] = np.where(
        keep, "fit", np.where(np.isin(np.arange(len(refs)), validation_indices),
                              "heldout_validation", "excluded")
    )
    audit["model_residual"] = np.nan
    check = audit["initial_reference_ok"].to_numpy(dtype=bool)
    audit.loc[check, "model_residual"] = y_all[check] - zero - slope * (x_all[check] - pivot)
    solution = {
        "zero_point": zero, "slope": float(slope), "color_pivot": pivot,
        "color_term_applied": bool(significant), "color_span": span,
        "n_candidates": int(refs["initial_reference_ok"].sum()),
        "n_used": int(keep.sum()), "n_holdout": int(len(validation_indices)),
        "holdout_mae_zero_only": holdout_constant_mae,
        "holdout_mae_with_color_term": holdout_color_mae,
        "zp_scatter_mad": robust_scatter,
    }
    return solution, audit


def calibrate_band(stars, image, band, calibration_matches, v_to_band=None,
                   min_references=6, aperture_radius=6, sky_in=10, sky_out=18):
    """Measure one band and derive robust Gaia GSPC zero point and color term."""
    band = band.upper()
    measured = measure_band_photometry(
        stars, image, band, v_to_band=v_to_band,
        aperture_radius=aperture_radius, sky_in=sky_in, sky_out=sky_out,
    )
    reference_table = select_calibration_references(measured, calibration_matches, band)
    solution, audit = fit_color_calibration(reference_table, min_references)
    measured[f"{band}_mag"] = measured[f"{band}_inst_mag"] + solution["zero_point"]
    solution["band"] = band
    print(f"{band}: {solution['n_used']} Gaia references, ZP={solution['zero_point']:.4f}, "
          f"MAD scatter={solution['zp_scatter_mad']:.4f} mag, "
          f"color slope={solution['slope']:+.4f} (applied={solution['color_term_applied']})")
    return measured, solution, audit


def calibrate_photometry(image_path, stars, band, calibration_matches,
                         reference_image=None, aperture_radius=6,
                         sky_in=10, sky_out=18, min_references=6):
    """Path-based reusable entry point for Gaia-calibrated V, B, or R photometry.

    B/R are registered against the V image. Returns the measured star table,
    calibration solution, reference-star audit, and registration diagnostics.
    """
    band = band.upper()
    if band not in ("B", "V", "R"):
        raise ValueError("Expected B, V, or R band")
    with fits.open(image_path) as hdul:
        image = hdul[0].data.astype(float)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D stacked FITS image: {image_path}")
    transform, registration = None, None
    if band != "V":
        if reference_image is None:
            raise ValueError("Provide the V reference image for B/R registration")
        transform, registration = estimate_band_registration(image, reference_image)

    measured, solution, refs = calibrate_band(
        stars, image, band, calibration_matches, v_to_band=transform,
        min_references=min_references,
        aperture_radius=aperture_radius, sky_in=sky_in, sky_out=sky_out,
    )
    return measured, solution, refs, registration


def finalize_bv_color_terms(stars, b_solution, v_solution):
    """Solve B-V from instrumental magnitudes, then apply both linear color terms."""
    result = stars.copy()
    b0 = result["B_inst_mag"] + b_solution["zero_point"]
    v0 = result["V_inst_mag"] + v_solution["zero_point"]
    sb, sv = b_solution["slope"], v_solution["slope"]
    pb, pv = b_solution["color_pivot"], v_solution["color_pivot"]
    denominator = 1 - sb + sv
    if abs(denominator) < 0.2:
        raise RuntimeError("B/V color transformation unstable; review filter calibration.")
    measured_bv = (b0 - v0 - sb * pb + sv * pv) / denominator
    result["B_mag"] = b0 + sb * (measured_bv - pb)
    result["V_mag"] = v0 + sv * (measured_bv - pv)
    return result


def apply_r_color_term(stars, r_solution):
    """Apply R color correction using the calibrated B-V of each science star."""
    result = stars.copy()
    color_bv = result["B_mag"] - result["V_mag"]
    result["R_mag"] = result["R_inst_mag"] + r_solution["zero_point"] + (
        r_solution["slope"] * (color_bv - r_solution["color_pivot"])
    )
    return result


def save_calibration_audit(band, solution, audit):
    """Store the used/rejected Gaia sources and the fitted band calibration."""
    prefix = CALIBRATION_OUTPUT_DIR / f"{CALIBRATION_PREFIX}_{band.lower()}"
    audit.to_csv(str(prefix) + "_calibration_refs.csv", index=False)
    with open(str(prefix) + "_calibration_summary.json", "w", encoding="utf-8") as handle:
        json.dump(solution, handle, indent=2)
    # Show the filter color-term check, including held-out and rejected references.
    good = audit["initial_reference_ok"].fillna(False).to_numpy(dtype=bool)
    used = audit["calibration_used"].fillna(False).to_numpy(dtype=bool)
    held = audit["calibration_role"].eq("heldout_validation").to_numpy(dtype=bool)
    excluded = good & ~used & ~held
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for mask, label, style in [
        (used, "Calibration fit", "o"),
        (held, "Independent validation", "s"),
        (excluded, "Rejected reference", "x"),
    ]:
        if np.any(mask):
            ax.scatter(audit.loc[mask, "catalog_BV"],
                       audit.loc[mask, "zp_individual"],
                       marker=style, s=24, alpha=0.8, label=label)
    if np.any(used):
        xmin = float(audit.loc[used, "catalog_BV"].min())
        xmax = float(audit.loc[used, "catalog_BV"].max())
        span = np.linspace(xmin, xmax, 100)
        ax.plot(span, solution["zero_point"] + solution["slope"] *
                (span - solution["color_pivot"]), label="Adopted calibration")
    ax.set(xlabel="Gaia DR3 synthetic B−V (mag)",
           ylabel=f"{band} catalog − instrumental (mag)",
           title=f"{band} photometric zero point and color term")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(str(prefix) + "_calibration_diagnostic.png", dpi=160)
    plt.close(fig)

### 8.5 Run V calibration

The V calibration returns a **provisional V magnitude** after applying the
robust zero point. Once B is measured in Section 9, any validated B/V color
terms are applied consistently to *all* stars using their measured B−V.
The Gaia reference table and calibration summary are saved for auditing.

In [ ]:
selected, v_calibration, v_reference_audit, _ = calibrate_photometry(
    V_STACK, selected, "V", calibration_matches,
    aperture_radius=APERTURE_RADIUS, sky_in=SKY_INNER, sky_out=SKY_OUTER,
    min_references=MIN_CALIBRATION_REFERENCES,
)
save_calibration_audit("V", v_calibration, v_reference_audit)
v_zero_point = v_calibration["zero_point"]
selected.to_csv(OUTPUT, index=False)
print(f"Saved {len(selected)} stars with provisional V photometry to {OUTPUT}")
display(selected[["StarID", "V_x_centroid", "V_y_centroid", "V_flux", "V_mag"]].head())

## 9. Calibrate B and Finalize B/V Magnitudes

The B image is automatically registered against V using matched stellar
patterns (`astroalign.find_transform`). The transform maps **B → V**, so its
inverse maps V-star coordinates to the B-image pixels. Measuring flux on the
original B image avoids interpolation errors caused by image resampling.
The registration RMS and matched-star count are checked before calibration.

The same `calibrate_band()` helpers obtain Gaia synthetic B magnitudes, reject
inconsistent references, test for a B−V color term, and calculate the B zero
point. If either B or V has a validated color term, both magnitudes are
corrected consistently using each science star's measured B−V.

In [ ]:
selected, b_calibration, b_reference_audit, b_registration = calibrate_photometry(
    B_STACK, selected, "B", calibration_matches, reference_image=data,
    aperture_radius=APERTURE_RADIUS, sky_in=SKY_INNER, sky_out=SKY_OUTER,
    min_references=MIN_CALIBRATION_REFERENCES,
)
print("B → V registration:", b_registration)
selected = finalize_bv_color_terms(selected, b_calibration, v_calibration)
save_calibration_audit("B", b_calibration, b_reference_audit)
b_zero_point = b_calibration["zero_point"]
selected.to_csv(OUTPUT, index=False)
print(f"Saved {len(selected)} stars with final B/V photometry to {OUTPUT}")
display(selected[["StarID", "B_mag", "V_mag", "B_flux", "V_flux"]].head())

## 10. Filter and Calculate Corrected Photometry

This section requires valid B and V magnitudes, applies the adopted reddening correction, and saves the CMD-ready star table. The optional magnitude-limit lines are currently disabled.


In [ ]:
BMV_ReddenAdjustment = 0.25
V_ReddenAdjustment = 0.25 * 3.2
filtered = selected[
    selected["B_mag"].notna()
    & selected["V_mag"].notna()
    # & (selected["B_mag"] <= 18)
    # & (selected["V_mag"] <= 18)
].copy()

filtered.to_csv(OUTPUT_FILTERED, index=False)
print(f"Saved {len(filtered)} filtered stars to {OUTPUT_FILTERED}")

calculated = filtered.copy()
calculated["BMV"] = calculated["B_mag"] - calculated["V_mag"]
calculated["BMV_Corrected"] = calculated["BMV"] - BMV_ReddenAdjustment
calculated["V_Corrected"] = calculated["V_mag"] - V_ReddenAdjustment

# calculated = calculated[(calculated["BMV_Corrected"] > 1.0)
#                         & (calculated["BMV_Corrected"] < 3.0)] # filter out outliers
calculated = calculated[
    calculated["BMV_Corrected"].between(-0.4, 2.0)
].copy()

calculated.to_csv(OUTPUT_CALCED, index=False)
print(f"Saved {len(calculated)} calculated stars to {OUTPUT_CALCED}")

calculated[
    ["StarID", "B_mag", "V_mag", "BMV", "BMV_Corrected", "V_Corrected"]
].head()


## 11. Estimate Distance and Attempt Age Measurement with a ZAMS Overlay

This section uses `calculated` dataframe before color-color outlier filtering. The ZAMS is shifted vertically to estimate the distance modulus and distance. At this stage, the unfiltered CMD does not show a sufficiently clear main-sequence turnoff, so the age is reported as unavailable rather than estimated.


In [ ]:
# Approximate empirical ZAMS table: intrinsic (B-V)_0 and absolute V magnitude M_V.
# Values are intentionally low-resolution because this section is a heuristic overlay,
# not a replacement for a full theoretical isochrone fit.
zams = pd.DataFrame({
    "BMV_Corrected": [
        -0.33, -0.30, -0.25, -0.20, -0.15, -0.10, -0.05,
         0.00,  0.10,  0.20,  0.30,  0.40,  0.50,  0.60,
         0.70,  0.80,  0.90,  1.00,  1.10,  1.20,  1.30,
         1.40,  1.50,  1.60,  1.70,  1.80
    ],
    "M_V": [
        -5.8, -4.5, -3.2, -2.2, -1.3, -0.6, -0.1,
         0.6,  1.2,  1.8,  2.3,  2.8,  3.4,  4.0,
         4.6,  5.2,  5.8,  6.4,  7.0,  7.6,  8.3,
         9.0,  9.7, 10.4, 11.1, 11.8
    ],
})

cmd = calculated[["BMV_Corrected", "V_Corrected"]].dropna().copy()
cmd = cmd[
    cmd["BMV_Corrected"].between(zams["BMV_Corrected"].min(),
                                  zams["BMV_Corrected"].max())
].copy()

if len(cmd) < 10:
    raise RuntimeError("Too few stars lie within the ZAMS color range.")

# Interpolate the ZAMS absolute magnitude at every observed color.
cmd["ZAMS_MV"] = np.interp(
    cmd["BMV_Corrected"], zams["BMV_Corrected"], zams["M_V"]
)
cmd["IndividualDistanceModulus"] = cmd["V_Corrected"] - cmd["ZAMS_MV"]

# Main-sequence binaries and evolved stars tend to lie above the ZAMS.
# A high quantile therefore gives a practical initial vertical alignment.
DISTANCE_MODULUS_QUANTILE = 0.70
fitted_distance_modulus = cmd["IndividualDistanceModulus"].quantile(
    DISTANCE_MODULUS_QUANTILE
)

# Set this to a number after visually inspecting the overlay, or leave as None
# to use the automatic estimate above.
MANUAL_DISTANCE_MODULUS = None
distance_modulus = (
    fitted_distance_modulus
    if MANUAL_DISTANCE_MODULUS is None
    else float(MANUAL_DISTANCE_MODULUS)
)

distance_pc = 10 ** ((distance_modulus + 5.0) / 5.0)
distance_ly = distance_pc * 3.26156

print(f"Automatic ZAMS distance modulus: {fitted_distance_modulus:.2f} mag")
print(f"Adopted distance modulus:        {distance_modulus:.2f} mag")
print(f"Estimated distance:              {distance_pc:,.0f} pc")
print(f"                                 {distance_ly:,.0f} light-years")

# Plot the corrected CMD and vertically shifted ZAMS.
plt.figure(figsize=(10, 8))
plt.scatter(
    calculated["BMV_Corrected"],
    calculated["V_Corrected"],
    s=24,
    alpha=0.65,
    label="Observed stars",
)
plt.plot(
    zams["BMV_Corrected"],
    zams["M_V"] + distance_modulus,
    linewidth=2.5,
    label=f"Shifted ZAMS (mu0 = {distance_modulus:.2f})",
)
plt.xlabel(r"Corrected color $(B-V)_0$")
plt.ylabel(r"Corrected magnitude $V_0$")
plt.title("NGC 2477 CMD with ZAMS Overlay")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.savefig(CMD_PLOT, dpi=300, bbox_inches="tight")

# At this stage the CMD has not yet gone through color-color outlier filtering.
estimated_age_gyr = np.nan
print(
    "Age not estimated reliably before color-color outlier filtering: "
    "the main-sequence turnoff is not sufficiently clear."
)

zams_result = pd.DataFrame([{
    "DistanceModulus": distance_modulus,
    "Distance_pc": distance_pc,
    "Distance_light_years": distance_ly,
    "EstimatedAge_Gyr": estimated_age_gyr,
}])

# Display the 20 bluest stars in the current unfiltered calculated dataframe.
bluest_stars = calculated.nsmallest(20, "BMV_Corrected")[
    [
        "StarID",
        "V_x_centroid",
        "V_y_centroid",
        "B_mag",
        "V_mag",
        "BMV",
        "BMV_Corrected",
        "V_Corrected",
    ]
]

display(zams_result)
display(bluest_stars)


## 12. R-Band Calibration and Color–Color Screening

### 12.1 Measure and calibrate R

The R image is registered against V using `astroalign`, and R-band aperture
photometry is measured at transformed V-star positions. The same reusable
Gaia DR3 reference matching and calibration helpers determine the R zero point
and test a color-dependent R correction against synthetic Johnson–Kron–Cousins
photometry. The R correction uses each star's observed, calibrated B−V; no
Gaia synthetic measurement is required for a science star.

### 12.2 Color–color outlier screening

CCS compares corrected B−V against V−R. It detects **color–color outliers**,
not cluster membership. The existing quadratic-locus/MAD screening is left
unchanged in this update; its separate model-selection redesign is a later step.

In [ ]:
# The same registration/calibration helper now handles R.
COLOR_LOCUS_SIGMA = 3.5
selected, r_calibration, r_reference_audit, r_registration = calibrate_photometry(
    R_STACK, selected, "R", calibration_matches, reference_image=data,
    aperture_radius=APERTURE_RADIUS, sky_in=SKY_INNER, sky_out=SKY_OUTER,
    min_references=MIN_CALIBRATION_REFERENCES,
)
print("R → V registration:", r_registration)
selected = apply_r_color_term(selected, r_calibration)
save_calibration_audit("R", r_calibration, r_reference_audit)
r_zero_point = r_calibration["zero_point"]
selected.to_csv(OUTPUT, index=False)

# Join the R measurements onto the already corrected CMD sample by StarID.
r_columns = ["StarID", "R_x_centroid", "R_y_centroid", "R_position_valid",
             "R_flux", "R_inst_mag", "R_mag"]
r_phot = calculated.merge(selected[r_columns], on="StarID", how="left", validate="one_to_one")
r_phot["VR"] = r_phot["V_mag"] - r_phot["R_mag"]

# Existing Section 12 CCS code, with the R-band calibration replaced above.
# Use stars with valid B-V and V-R measurements to fit the dominant locus.

fit_mask = (
    np.isfinite(r_phot["BMV_Corrected"])
    & np.isfinite(r_phot["VR"])
)

fit_data = r_phot.loc[fit_mask].copy()

if len(fit_data) < 20:
    raise RuntimeError(
        "Too few valid R measurements to fit the color-color locus. "
        "Check R image registration and the photometric calibration."
    )

# Iteratively fit a quadratic color-color locus and remove strong outliers.
keep = np.ones(len(fit_data), dtype=bool)

for _ in range(4):
    x = fit_data.loc[keep, "BMV_Corrected"].to_numpy()
    y = fit_data.loc[keep, "VR"].to_numpy()

    coeff = np.polyfit(x, y, deg=2)
    predicted = np.polyval(coeff, fit_data["BMV_Corrected"].to_numpy())
    residual = fit_data["VR"].to_numpy() - predicted

    center = np.median(residual[keep])
    mad = np.median(np.abs(residual[keep] - center))
    robust_sigma = 1.4826 * mad

    if not np.isfinite(robust_sigma) or robust_sigma == 0:
        break

    keep = np.abs(residual - center) <= COLOR_LOCUS_SIGMA * robust_sigma

# Final residuals for every star with valid R photometry.
r_phot["VR_locus_expected"] = np.polyval(
    coeff,
    r_phot["BMV_Corrected"],
)

r_phot["VR_locus_residual"] = (
    r_phot["VR"] - r_phot["VR_locus_expected"]
)

final_center = np.median(residual[keep])
final_mad = np.median(np.abs(residual[keep] - final_center))
final_sigma = 1.4826 * final_mad

r_phot["not_belong_flag"] = (
    ~r_phot["R_position_valid"]
    | ~np.isfinite(r_phot["R_flux"])
    | (r_phot["R_flux"] <= 0)
    | (
        np.abs(r_phot["VR_locus_residual"] - final_center)
        > COLOR_LOCUS_SIGMA * final_sigma
    )
)

not_belong = r_phot.loc[
    r_phot["not_belong_flag"],
    [
        "StarID",
        "V_x_centroid",
        "V_y_centroid",
    ],
].copy()

print(f"Valid R measurements: {fit_mask.sum()} / {len(r_phot)}")
print(f"CCS color–color outlier candidates: {len(not_belong)}")
print(not_belong.shape)
not_belong.head(20)
not_belong.to_csv(NOT_BELONG, index=False)

## 13. Remove Photometric Non-Member Candidates and Regenerate the CMD

This section saves the flagged stars to `NOT_BELONG`, removes matching stars from `calculated`, and redraws the corrected CMD.

The matching is performed by `StarID`, while the saved CSV includes `V_x_centroid` and `V_y_centroid`.


In [ ]:
# Save stars flagged by the B-V / V-R color-color test.
not_belong.to_csv(NOT_BELONG, index=False)
print(f"Saved {len(not_belong)} stars to {NOT_BELONG}")

# Find stars from calculated that were flagged as not belonging.
not_belong_ids = set(not_belong["StarID"])

removed_from_calculated = calculated[
    calculated["StarID"].isin(not_belong_ids)
].copy()

print(
    f"Stars found in calculated and removed: "
    f"{len(removed_from_calculated)}"
)

# Remove flagged stars from calculated.
calculated = calculated[
    ~calculated["StarID"].isin(not_belong_ids)
].copy()

# Save the updated calculated table.
calculated.to_csv(OUTPUT_CALCED, index=False)
print(f"Remaining CMD stars: {len(calculated)}")
print(f"Updated calculated data saved to {OUTPUT_CALCED}")

# Regenerate the corrected CMD.
plt.figure(figsize=(10, 8))

plt.scatter(
    calculated["BMV_Corrected"],
    calculated["V_Corrected"],
    s=25,
    alpha=0.8,
)

plt.xlabel("BMV_Corrected")
plt.ylabel("V_Corrected")
plt.title("NGC 2477 Corrected CMD After B-V / V-R Screening")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

removed_from_calculated[
    [
        "StarID",
        "V_x_centroid",
        "V_y_centroid",
        "BMV_Corrected",
        "V_Corrected",
    ]
].head(20)


## 14. Spectal Class
### Approximate Spectral Class from Corrected B−V Color

| Spectral Class | B−V Corrected Range |
|---------------|--------------------:|
| O | < -0.30 |
| B | -0.30 to < 0.00 |
| A | 0.00 to < 0.35 |
| F | 0.35 to < 0.60 |
| G | 0.60 to < 0.85 |
| K | 0.85 to < 1.30 |
| M | ≥ 1.30 |

This classification is an approximate mapping from corrected B−V color to the Morgan–Keenan (MK) spectral classes. It is intended for visualization and heuristic analysis rather than precise stellar classification.


In [ ]:
def spectral_class_from_bmv(bmv):
    if pd.isna(bmv):
        return ""
    if bmv < -0.3:
        return "O"
    elif bmv < 0:
        return "B"
    elif bmv < 0.35:
        return "A"
    elif bmv < 0.6:
        return "F"
    elif bmv < 0.85:
        return "G"
    elif bmv < 1.3:
        return "K"
    else:
        return "M"

calculated["SpectralClass"] = calculated["BMV_Corrected"].apply(
    spectral_class_from_bmv
)
calculated.to_csv(OUTPUT_CALCED, index=False)

## 15. Estimate Distance and Age with a ZAMS Overlay

A Zero-Age Main Sequence (ZAMS) gives the expected absolute magnitude, $M_V$, for main-sequence stars of each intrinsic color, $(B-V)_0$. The code shifts the ZAMS vertically until it best overlaps the observed main sequence. This vertical shift is the distance modulus:

$$\mu_0 = V_0-M_V, \qquad d=10^{(\mu_0+5)/5}\ \mathrm{pc}.
$$

The cluster age is estimated separately from the main-sequence turnoff. The brightest plausible turnoff star is converted to an approximate stellar mass and main-sequence lifetime. This is a heuristic estimate; reliable age measurement normally requires fitting theoretical isochrones. If the CMD does not contain sufficiently blue main-sequence stars, the code reports that the age cannot be estimated reliably.


In [ ]:
# Approximate empirical ZAMS table: intrinsic (B-V)_0 and absolute V magnitude M_V.
# Values are intentionally low-resolution because this section is a heuristic overlay,
# not a replacement for a full theoretical isochrone fit.
zams = pd.DataFrame({
    "BMV_Corrected": [
        -0.33, -0.30, -0.25, -0.20, -0.15, -0.10, -0.05,
         0.00,  0.10,  0.20,  0.30,  0.40,  0.50,  0.60,
         0.70,  0.80,  0.90,  1.00,  1.10,  1.20,  1.30,
         1.40,  1.50,  1.60,  1.70,  1.80
    ],
    "M_V": [
        -5.8, -4.5, -3.2, -2.2, -1.3, -0.6, -0.1,
         0.6,  1.2,  1.8,  2.3,  2.8,  3.4,  4.0,
         4.6,  5.2,  5.8,  6.4,  7.0,  7.6,  8.3,
         9.0,  9.7, 10.4, 11.1, 11.8
    ],
})

cmd = calculated[["BMV_Corrected", "V_Corrected"]].dropna().copy()
cmd = cmd[
    cmd["BMV_Corrected"].between(zams["BMV_Corrected"].min(),
                                  zams["BMV_Corrected"].max())
].copy()

if len(cmd) < 10:
    raise RuntimeError("Too few stars lie within the ZAMS color range.")

# Interpolate the ZAMS absolute magnitude at every observed color.
cmd["ZAMS_MV"] = np.interp(
    cmd["BMV_Corrected"], zams["BMV_Corrected"], zams["M_V"]
)
cmd["IndividualDistanceModulus"] = cmd["V_Corrected"] - cmd["ZAMS_MV"]

# Main-sequence binaries and evolved stars tend to lie above the ZAMS.
# A high quantile therefore gives a practical initial vertical alignment.
DISTANCE_MODULUS_QUANTILE = 0.70
fitted_distance_modulus = cmd["IndividualDistanceModulus"].quantile(
    DISTANCE_MODULUS_QUANTILE
)

# Set this to a number after visually inspecting the overlay, or leave as None
# to use the automatic estimate above.
MANUAL_DISTANCE_MODULUS = None
distance_modulus = (
    fitted_distance_modulus
    if MANUAL_DISTANCE_MODULUS is None
    else float(MANUAL_DISTANCE_MODULUS)
)

distance_pc = 10 ** ((distance_modulus + 5.0) / 5.0)
distance_ly = distance_pc * 3.26156

print(f"Automatic ZAMS distance modulus: {fitted_distance_modulus:.2f} mag")
print(f"Adopted distance modulus:        {distance_modulus:.2f} mag")
print(f"Estimated distance:              {distance_pc:,.0f} pc")
print(f"                                 {distance_ly:,.0f} light-years")

# Plot the corrected CMD and vertically shifted ZAMS.
plt.figure(figsize=(10, 8))
plt.scatter(
    calculated["BMV_Corrected"],
    calculated["V_Corrected"],
    s=24,
    alpha=0.65,
    label="Observed stars",
)
plt.plot(
    zams["BMV_Corrected"],
    zams["M_V"] + distance_modulus,
    linewidth=2.5,
    label=f"Shifted ZAMS (mu0 = {distance_modulus:.2f})",
)
plt.xlabel(r"Corrected color $(B-V)_0$")
plt.ylabel(r"Corrected magnitude $V_0$")
plt.title("NGC 2477 CMD with ZAMS Overlay")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.savefig(CMD_PLOT, dpi=300, bbox_inches="tight")

# Heuristic main-sequence-turnoff age estimate.
# Restrict the search to relatively blue stars close to, or above, the shifted ZAMS.
cmd["Absolute_V"] = cmd["V_Corrected"] - distance_modulus
cmd["DeltaFromZAMS"] = cmd["Absolute_V"] - cmd["ZAMS_MV"]
turnoff_candidates = cmd[
    (cmd["BMV_Corrected"] <= 0.60)
    & (cmd["DeltaFromZAMS"].between(-1.5, 0.8))
].copy()

if len(turnoff_candidates) < 3:
    estimated_age_gyr = np.nan
    print(
        "Age not estimated reliably: the CMD contains too few plausible "
        "blue main-sequence/turnoff stars."
    )
else:
    # Use the bright end of the plausible turnoff population rather than one star.
    turnoff_mv = turnoff_candidates["Absolute_V"].quantile(0.20)
    turnoff_bmv = turnoff_candidates.loc[
        (turnoff_candidates["Absolute_V"] - turnoff_mv).abs().idxmin(),
        "BMV_Corrected",
    ]

    # Approximate conversion: M_V -> luminosity -> mass -> main-sequence lifetime.
    M_V_SUN = 4.83
    turnoff_luminosity = 10 ** (0.4 * (M_V_SUN - turnoff_mv))
    turnoff_mass = turnoff_luminosity ** (1.0 / 3.5)
    estimated_age_gyr = 10.0 * turnoff_mass ** (-2.5)

    print(f"Approximate turnoff (B-V)_0:     {turnoff_bmv:.2f}")
    print(f"Approximate turnoff M_V:         {turnoff_mv:.2f}")
    print(f"Approximate turnoff mass:        {turnoff_mass:.2f} solar masses")
    print(f"Heuristic cluster age:           {estimated_age_gyr:.2f} Gyr")

zams_result = pd.DataFrame([{
    "DistanceModulus": distance_modulus,
    "Distance_pc": distance_pc,
    "Distance_light_years": distance_ly,
    "EstimatedAge_Gyr": estimated_age_gyr,
}])

zams_result


# 16. Gaia Astrometric Membership Analysis

The V-image celestial WCS was solved and validated in Section 8 for Gaia
synthetic-photometry calibration. Here it is reused to query **all Gaia DR3
sources with astrometric measurements**, including sources absent from GSPC.
This is essential: the photometric calibration catalog must not restrict Gaia
membership selection. The subsequent astrometric selection remains unchanged.

## 16.1 — Query Gaia around the NGC 2477 field

The Gaia search region is derived automatically from the solved V-image WCS rather than from hard-coded NGC 2477 coordinates.

The four V-image corners are transformed to sky coordinates. Their spherical mean defines the query center, and the distance to the farthest corner defines the cone-search radius with a small margin.

The query requests Gaia DR3 astrometry and photometry needed for later membership analysis:

- `source_id`
- RA and Dec
- parallax and parallax uncertainty
- proper motion in RA and Dec and their uncertainties
- RUWE
- Gaia G, BP, and RP magnitudes

No cluster-membership cuts are applied at this stage; this is the field catalog.


In [ ]:
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia

# Determine the V-image footprint from the astrometric solution.
ny, nx = original_v_shape

corner_x = np.array([0.0, nx - 1.0, nx - 1.0, 0.0])
corner_y = np.array([0.0, 0.0, ny - 1.0, ny - 1.0])

corner_sky = astrometric_wcs.pixel_to_world(corner_x, corner_y)

# Use the mean unit-vector direction as a robust spherical center.
xyz = corner_sky.cartesian.xyz.value
mean_xyz = xyz.mean(axis=1)
mean_xyz /= np.linalg.norm(mean_xyz)

gaia_query_center = SkyCoord(
    x=mean_xyz[0],
    y=mean_xyz[1],
    z=mean_xyz[2],
    representation_type="cartesian",
    frame="icrs",
)

# Radius = farthest corner + 5% margin.
corner_separations = gaia_query_center.separation(corner_sky)
gaia_query_radius = 1.05 * corner_separations.max()

# Convert the Cartesian SkyCoord representation back to spherical RA/Dec.
gaia_center_ra = gaia_query_center.spherical.lon.deg
gaia_center_dec = gaia_query_center.spherical.lat.deg

print(
    f"Gaia query center: "
    f"RA={gaia_center_ra:.6f} deg, "
    f"Dec={gaia_center_dec:.6f} deg"
)

print(
    f"Gaia query radius: "
    f"{gaia_query_radius.to(u.arcmin).value:.3f} arcmin"
)

ra0 = gaia_center_ra
dec0 = gaia_center_dec
radius_deg = gaia_query_radius.to(u.deg).value

gaia_adql = f'''
SELECT
    source_id,
    ra,
    dec,
    parallax,
    parallax_error,
    pmra,
    pmra_error,
    pmdec,
    pmdec_error,
    ruwe,
    astrometric_params_solved,
    visibility_periods_used,
    phot_g_mean_mag,
    phot_bp_mean_mag,
    phot_rp_mean_mag,
    bp_rp
FROM gaiadr3.gaia_source
WHERE 1 = CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', {ra0:.10f}, {dec0:.10f}, {radius_deg:.10f})
)
'''

gaia_job = Gaia.launch_job_async(gaia_adql)
gaia_table = gaia_job.get_results()
gaia_field = gaia_table.to_pandas()

print(f"Gaia DR3 sources returned: {len(gaia_field):,}")
display(gaia_field.head())


## 16.2 — Cross-match the photometric stars to Gaia spatially

The starting catalog is reloaded from `OUTPUT_FILTERED`, before the later color-color non-member screening.

For each star:

1. take the native V-image pixel positions `V_x_centroid`, `V_y_centroid`;
2. transform these pixel positions directly to RA/Dec using the solved V-image WCS;
3. find the nearest Gaia DR3 source;
4. accept the match only if it lies within `GAIA_MATCH_RADIUS_ARCSEC`.

A star with no Gaia source inside the allowed radius remains unmatched rather than being classified as a non-member.


In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

GAIA_MATCH_RADIUS_ARCSEC = 1.5

gaia_photometry = pd.read_csv(OUTPUT_FILTERED).copy()

required_cols = ["StarID", "V_x_centroid", "V_y_centroid"]
missing_cols = [c for c in required_cols if c not in gaia_photometry.columns]
if missing_cols:
    raise KeyError(
        f"OUTPUT_FILTERED is missing required columns: {missing_cols}"
    )

# Native V-image pixel positions are already in the coordinate convention
# expected by astrometric_wcs.pixel_to_world (zero-based NumPy/Photutils pixels).
v_x_data = gaia_photometry["V_x_centroid"].to_numpy(dtype=float)
v_y_data = gaia_photometry["V_y_centroid"].to_numpy(dtype=float)

# Convert image pixel coordinates to celestial coordinates.
phot_sky = astrometric_wcs.pixel_to_world(v_x_data, v_y_data)

gaia_photometry["RA_deg"] = phot_sky.ra.deg
gaia_photometry["Dec_deg"] = phot_sky.dec.deg

# Build Gaia sky-coordinate catalog.
gaia_sky = SkyCoord(
    ra=gaia_field["ra"].to_numpy(dtype=float) * u.deg,
    dec=gaia_field["dec"].to_numpy(dtype=float) * u.deg,
    frame="icrs",
)

# Nearest-neighbor spherical match.
idx, sep2d, _ = phot_sky.match_to_catalog_sky(gaia_sky)
matched = sep2d <= GAIA_MATCH_RADIUS_ARCSEC * u.arcsec

gaia_photometry["GaiaMatched"] = matched
gaia_photometry["GaiaMatchSepArcsec"] = sep2d.arcsec

# Prepare Gaia columns in the photometry table.
gaia_columns = [
    "source_id",
    "ra",
    "dec",
    "parallax",
    "parallax_error",
    "pmra",
    "pmra_error",
    "pmdec",
    "pmdec_error",
    "ruwe",
    "astrometric_params_solved",
    "visibility_periods_used",
    "phot_g_mean_mag",
    "phot_bp_mean_mag",
    "phot_rp_mean_mag",
    "bp_rp",
]

for col in gaia_columns:
    gaia_photometry[f"Gaia_{col}"] = np.nan

matched_rows = np.where(matched)[0]
matched_gaia_rows = idx[matched]

for col in gaia_columns:
    values = gaia_field.iloc[matched_gaia_rows][col].to_numpy()
    gaia_photometry.loc[
        gaia_photometry.index[matched_rows],
        f"Gaia_{col}"
    ] = values

print(
    f"Gaia matches within {GAIA_MATCH_RADIUS_ARCSEC:.2f} arcsec: "
    f"{matched.sum():,} / {len(gaia_photometry):,}"
)
print(
    f"Median accepted match separation: "
    f"{gaia_photometry.loc[gaia_photometry['GaiaMatched'], 'GaiaMatchSepArcsec'].median():.3f} arcsec"
)

display(
    gaia_photometry[
        [
            "StarID",
            "V_x_centroid",
            "V_y_centroid",
            "RA_deg",
            "Dec_deg",
            "GaiaMatched",
            "GaiaMatchSepArcsec",
            "Gaia_source_id",
            "Gaia_parallax",
            "Gaia_pmra",
            "Gaia_pmdec",
            "Gaia_ruwe",
        ]
    ].head(20)
)


## 16.3 — Determine the NGC 2477 astrometric concentration

Probable members of an open cluster should occupy a compact concentration in the three-dimensional Gaia astrometric space:

$$
(\varpi,\ \mu_{\alpha *},\ \mu_\delta)
$$

where `parallax` is $$\varpi$$, `pmra` is $$\mu_{\alpha *}$$, and `pmdec` is $$\mu_\delta$$.

The earlier version incorrectly assumed that the **largest** DBSCAN component in the full Gaia field must be NGC 2477. In this crowded Galactic field, the largest component can instead be a broad field-star population.

The revised approach searches several DBSCAN density scales and selects the **most compact, dense, sufficiently populated** astrometric component rather than the largest component. The selected clump is then summarized with robust medians and MAD-based scatter estimates.

This remains an automatic, data-driven first-pass membership estimate; no published NGC 2477 parallax or proper motion is hard-coded.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

astro = gaia_field.copy()

astro_mask = (
    np.isfinite(astro["parallax"])
    & np.isfinite(astro["pmra"])
    & np.isfinite(astro["pmdec"])
    & np.isfinite(astro["ruwe"])
    & (astro["ruwe"] < 1.4)
    & np.isfinite(astro["visibility_periods_used"])
    & (astro["visibility_periods_used"] >= 8)
)

astro_clean = astro.loc[astro_mask].copy()

if len(astro_clean) < 30:
    raise RuntimeError(
        "Too few clean Gaia sources are available to identify an astrometric concentration."
    )

feature_cols = ["parallax", "pmra", "pmdec"]
X = astro_clean[feature_cols].to_numpy(dtype=float)

# Robust field-scale standardization.
feature_median = np.median(X, axis=0)
feature_mad = np.median(np.abs(X - feature_median), axis=0)
feature_scale = 1.4826 * feature_mad

if np.any(~np.isfinite(feature_scale)) or np.any(feature_scale <= 0):
    raise RuntimeError("Could not compute valid robust scales for Gaia astrometry.")

X_scaled = (X - feature_median) / feature_scale

def robust_center_sigma(values):
    values = np.asarray(values, dtype=float)
    center = np.median(values)
    mad = np.median(np.abs(values - center))
    sigma = 1.4826 * mad
    return center, sigma

# Search several density scales.  The real cluster should be a compact
# concentration, whereas the Galactic field tends to form much broader groups.
DBSCAN_EPS_CANDIDATES = [0.08, 0.10, 0.12, 0.15, 0.18, 0.22, 0.28]
DBSCAN_MIN_SAMPLES = 10
MIN_CLUSTER_SIZE = 25
MAX_CLUSTER_FRACTION = 0.40

cluster_candidates = []

for eps in DBSCAN_EPS_CANDIDATES:
    labels = DBSCAN(
        eps=eps,
        min_samples=DBSCAN_MIN_SAMPLES
    ).fit_predict(X_scaled)

    for label in np.unique(labels):
        if label < 0:
            continue

        mask = labels == label
        n = int(mask.sum())

        if n < MIN_CLUSTER_SIZE:
            continue

        # Reject a component containing a very large fraction of the entire
        # Gaia field; that is much more likely to be the broad field population.
        if n > MAX_CLUSTER_FRACTION * len(astro_clean):
            continue

        component = astro_clean.loc[mask].copy()

        p_center, p_sigma = robust_center_sigma(component["parallax"])
        ra_center, ra_sigma = robust_center_sigma(component["pmra"])
        dec_center, dec_sigma = robust_center_sigma(component["pmdec"])

        if (
            not np.isfinite(p_sigma) or p_sigma <= 0
            or not np.isfinite(ra_sigma) or ra_sigma <= 0
            or not np.isfinite(dec_sigma) or dec_sigma <= 0
        ):
            continue

        # Compactness volume in physical astrometric units.
        # Floors prevent a tiny numerical scatter from creating an infinite score.
        volume = (
            max(p_sigma, 0.01)
            * max(ra_sigma, 0.03)
            * max(dec_sigma, 0.03)
        )

        # Favor compact groups while still rewarding a meaningful population size.
        density_score = np.sqrt(n) / volume

        cluster_candidates.append({
            "eps": eps,
            "label": int(label),
            "n": n,
            "score": density_score,
            "mask": mask,
            "parallax_center": p_center,
            "parallax_sigma": p_sigma,
            "pmra_center": ra_center,
            "pmra_sigma": ra_sigma,
            "pmdec_center": dec_center,
            "pmdec_sigma": dec_sigma,
        })

if not cluster_candidates:
    raise RuntimeError(
        "No compact Gaia astrometric concentration was found. "
        "Inspect the proper-motion/parallax diagrams and adjust the DBSCAN search parameters."
    )

best_cluster = max(cluster_candidates, key=lambda d: d["score"])

ngc2477_clump = astro_clean.loc[best_cluster["mask"]].copy()

NGC2477_PARALLAX_CENTER = best_cluster["parallax_center"]
NGC2477_PARALLAX_SIGMA = best_cluster["parallax_sigma"]
NGC2477_PMRA_CENTER = best_cluster["pmra_center"]
NGC2477_PMRA_SIGMA = best_cluster["pmra_sigma"]
NGC2477_PMDEC_CENTER = best_cluster["pmdec_center"]
NGC2477_PMDEC_SIGMA = best_cluster["pmdec_sigma"]

print(f"Clean Gaia sources used: {len(astro_clean):,}")
print(f"Selected DBSCAN eps: {best_cluster['eps']:.2f}")
print(f"Selected compact astrometric concentration size: {len(ngc2477_clump):,}")
print()
print("First-pass NGC 2477 astrometric concentration:")
print(
    f"  parallax = {NGC2477_PARALLAX_CENTER:.4f} "
    f"+/- {NGC2477_PARALLAX_SIGMA:.4f} mas (robust sigma)"
)
print(
    f"  pmRA     = {NGC2477_PMRA_CENTER:.4f} "
    f"+/- {NGC2477_PMRA_SIGMA:.4f} mas/yr"
)
print(
    f"  pmDec    = {NGC2477_PMDEC_CENTER:.4f} "
    f"+/- {NGC2477_PMDEC_SIGMA:.4f} mas/yr"
)

plt.figure(figsize=(9, 7))
plt.scatter(
    astro_clean["pmra"],
    astro_clean["pmdec"],
    s=12,
    alpha=0.20,
    label="Clean Gaia field",
)
plt.scatter(
    ngc2477_clump["pmra"],
    ngc2477_clump["pmdec"],
    s=25,
    alpha=0.85,
    label="Selected compact concentration",
)
plt.xlabel("Gaia pmRA (mas/yr)")
plt.ylabel("Gaia pmDec (mas/yr)")
plt.title("NGC 2477 Field: Gaia Proper-Motion Concentration")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 6))
plt.hist(
    astro_clean["parallax"],
    bins=60,
    alpha=0.30,
    label="Clean Gaia field",
)
plt.hist(
    ngc2477_clump["parallax"],
    bins=30,
    alpha=0.75,
    label="Selected compact concentration",
)
plt.xlabel("Gaia parallax (mas)")
plt.ylabel("Number of sources")
plt.title("NGC 2477 Field: Gaia Parallax Distribution")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 16.4 — Assign membership to the photometric stars using sigma cuts

For this initial implementation, membership is determined with independent sigma cuts in parallax, pmRA, and pmDec.

A matched star is considered a probable NGC 2477 member only when all three measurements lie within `GAIA_MEMBERSHIP_SIGMA` robust standard deviations of the astrometric concentration found in Step 7.


In [ ]:
GAIA_MEMBERSHIP_SIGMA = 3.0

has_valid_gaia_astrometry = (
    gaia_photometry["GaiaMatched"].fillna(False)
    & np.isfinite(gaia_photometry["Gaia_parallax"])
    & np.isfinite(gaia_photometry["Gaia_pmra"])
    & np.isfinite(gaia_photometry["Gaia_pmdec"])
)

for name, sigma in [
    ("parallax", NGC2477_PARALLAX_SIGMA),
    ("pmRA", NGC2477_PMRA_SIGMA),
    ("pmDec", NGC2477_PMDEC_SIGMA),
]:
    if not np.isfinite(sigma) or sigma <= 0:
        raise RuntimeError(f"Invalid NGC 2477 robust sigma for {name}: {sigma}")

gaia_photometry["GaiaParallaxSigmaOffset"] = np.nan
gaia_photometry["GaiaPmraSigmaOffset"] = np.nan
gaia_photometry["GaiaPmdecSigmaOffset"] = np.nan

valid_idx = gaia_photometry.index[has_valid_gaia_astrometry]

gaia_photometry.loc[valid_idx, "GaiaParallaxSigmaOffset"] = (
    np.abs(gaia_photometry.loc[valid_idx, "Gaia_parallax"] - NGC2477_PARALLAX_CENTER)
    / NGC2477_PARALLAX_SIGMA
)
gaia_photometry.loc[valid_idx, "GaiaPmraSigmaOffset"] = (
    np.abs(gaia_photometry.loc[valid_idx, "Gaia_pmra"] - NGC2477_PMRA_CENTER)
    / NGC2477_PMRA_SIGMA
)
gaia_photometry.loc[valid_idx, "GaiaPmdecSigmaOffset"] = (
    np.abs(gaia_photometry.loc[valid_idx, "Gaia_pmdec"] - NGC2477_PMDEC_CENTER)
    / NGC2477_PMDEC_SIGMA
)

gaia_photometry["GaiaProbableMember"] = (
    has_valid_gaia_astrometry
    & (gaia_photometry["GaiaParallaxSigmaOffset"] <= GAIA_MEMBERSHIP_SIGMA)
    & (gaia_photometry["GaiaPmraSigmaOffset"] <= GAIA_MEMBERSHIP_SIGMA)
    & (gaia_photometry["GaiaPmdecSigmaOffset"] <= GAIA_MEMBERSHIP_SIGMA)
)

print(f"Sigma threshold: {GAIA_MEMBERSHIP_SIGMA:.1f}")
print(
    f"Probable Gaia members: {gaia_photometry['GaiaProbableMember'].sum():,} "
    f"/ {len(gaia_photometry):,}"
)

display(
    gaia_photometry[
        [
            "StarID", "GaiaMatched",
            "Gaia_parallax", "Gaia_pmra", "Gaia_pmdec",
            "GaiaParallaxSigmaOffset",
            "GaiaPmraSigmaOffset",
            "GaiaPmdecSigmaOffset",
            "GaiaProbableMember",
        ]
    ].head(30)
)


## 16.5 — Treat stars without Gaia matches as non-members

For this initial simplified pipeline:

- Gaia match + passes all sigma cuts → probable member
- Gaia match + fails one or more sigma cuts → non-member
- no Gaia match → non-member

This is intentionally simple. A later, more conservative version could keep unmatched stars in a separate unknown-membership category.


In [ ]:
gaia_photometry["GaiaMembershipStatus"] = "Non-member"

gaia_photometry.loc[
    gaia_photometry["GaiaProbableMember"],
    "GaiaMembershipStatus"
] = "Probable member"

matched_count = int(gaia_photometry["GaiaMatched"].fillna(False).sum())
unmatched_count = len(gaia_photometry) - matched_count
member_count = int(gaia_photometry["GaiaProbableMember"].sum())
nonmember_count = len(gaia_photometry) - member_count

print(f"Total photometric stars: {len(gaia_photometry):,}")
print(f"Gaia matched:            {matched_count:,}")
print(f"No Gaia match:           {unmatched_count:,}")
print(f"Probable members:        {member_count:,}")
print(f"Non-members:             {nonmember_count:,}")


## 16.6 — Keep Gaia probable members only

Create a Gaia-cleaned photometric catalog containing only probable NGC 2477 members. This preserves the original B/V photometry together with the Gaia astrometric measurements for later CMD construction and age/distance analysis.


In [ ]:
gaia_members = gaia_photometry[
    gaia_photometry["GaiaProbableMember"]
].copy()

GAIA_MEMBERS_OUTPUT = str(
    Path(OUTPUT_FILTERED).with_name("NGC2477_gaia_probable_members.csv")
)

gaia_members.to_csv(GAIA_MEMBERS_OUTPUT, index=False)

print(f"Gaia probable members saved: {len(gaia_members):,}")
print(f"Output file: {GAIA_MEMBERS_OUTPUT}")

display(
    gaia_members[
        [
            "StarID",
            "B_mag",
            "V_mag",
            "Gaia_source_id",
            "Gaia_parallax",
            "Gaia_pmra",
            "Gaia_pmdec",
            "GaiaMatchSepArcsec",
        ]
    ].head(30)
)


## 16.7 Estimate NGC 2477 Age and Distance Using Gaia Probable Members Only

This section applies the **same ZAMS distance and main-sequence-turnoff age methodology and the same parameter values used in Section 17**, changing only the input sample to Gaia probable members.

This makes the Section 17 vs. Section 18 comparison methodologically consistent.


In [ ]:
# Start from Gaia probable members only.
gaia_cmd = gaia_members.copy()

# Recompute the same corrected photometric quantities used earlier in the notebook.
gaia_cmd["BMV"] = gaia_cmd["B_mag"] - gaia_cmd["V_mag"]
gaia_cmd["BMV_Corrected"] = gaia_cmd["BMV"] - BMV_ReddenAdjustment
gaia_cmd["V_Corrected"] = gaia_cmd["V_mag"] - V_ReddenAdjustment

# Use the same physically plausible color range used earlier.
gaia_cmd = gaia_cmd[
    gaia_cmd["BMV_Corrected"].between(-0.4, 2.0)
    & np.isfinite(gaia_cmd["V_Corrected"])
].copy()

print(f"Gaia probable members available for CMD analysis: {len(gaia_cmd):,}")

if len(gaia_cmd) < 10:
    raise RuntimeError(
        "Too few Gaia probable members remain for a meaningful CMD age/distance estimate."
    )

plt.figure(figsize=(10, 8))
plt.scatter(
    gaia_cmd["BMV_Corrected"],
    gaia_cmd["V_Corrected"],
    s=28,
    alpha=0.8,
)
plt.xlabel(r"Corrected color $(B-V)_0$")
plt.ylabel(r"Corrected magnitude $V_0$")
plt.title("NGC 2477 CMD — Gaia Probable Members Only")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 16.7.1 Distance and age using the Section 17 methodology

The following calculation intentionally mirrors Section 17:

- same empirical ZAMS table;
- same `DISTANCE_MODULUS_QUANTILE = 0.70`;
- same main-sequence/turnoff color limit, `(B-V)_0 <= 0.60`;
- same `DeltaFromZAMS` interval, `[-1.5, 0.8]`;
- same turnoff absolute-magnitude quantile, `0.20`;
- same solar absolute magnitude, `M_V_SUN = 4.83`;
- same luminosity→mass exponent, `1/3.5`;
- same main-sequence lifetime relation, `10 * M^-2.5`.


In [ ]:
# Use exactly the same age/distance methodology and parameters as Section 17.
gaia_zams_cmd = gaia_cmd[
    gaia_cmd["BMV_Corrected"].between(
        zams["BMV_Corrected"].min(),
        zams["BMV_Corrected"].max()
    )
].copy()

if len(gaia_zams_cmd) < 10:
    raise RuntimeError("Too few Gaia members lie within the ZAMS color range.")

gaia_zams_cmd["ZAMS_MV"] = np.interp(
    gaia_zams_cmd["BMV_Corrected"],
    zams["BMV_Corrected"],
    zams["M_V"],
)
gaia_zams_cmd["IndividualDistanceModulus"] = (
    gaia_zams_cmd["V_Corrected"] - gaia_zams_cmd["ZAMS_MV"]
)

# SAME AS SECTION 17
DISTANCE_MODULUS_QUANTILE = 0.70
gaia_fitted_distance_modulus = gaia_zams_cmd[
    "IndividualDistanceModulus"
].quantile(DISTANCE_MODULUS_QUANTILE)

# SAME AS SECTION 17: automatic estimate unless explicitly overridden.
MANUAL_DISTANCE_MODULUS = None
gaia_distance_modulus = (
    gaia_fitted_distance_modulus
    if MANUAL_DISTANCE_MODULUS is None
    else float(MANUAL_DISTANCE_MODULUS)
)

gaia_distance_pc = 10 ** ((gaia_distance_modulus + 5.0) / 5.0)
gaia_distance_ly = gaia_distance_pc * 3.26156

print(f"Automatic ZAMS distance modulus: {gaia_fitted_distance_modulus:.2f} mag")
print(f"Adopted distance modulus:        {gaia_distance_modulus:.2f} mag")
print(f"Estimated distance:              {gaia_distance_pc:,.0f} pc")
print(f"                                 {gaia_distance_ly:,.0f} light-years")

# SAME AS SECTION 17: derive turnoff candidates relative to the shifted ZAMS.
gaia_zams_cmd["Absolute_V"] = (
    gaia_zams_cmd["V_Corrected"] - gaia_distance_modulus
)
gaia_zams_cmd["DeltaFromZAMS"] = (
    gaia_zams_cmd["Absolute_V"] - gaia_zams_cmd["ZAMS_MV"]
)

gaia_turnoff_candidates = gaia_zams_cmd[
    (gaia_zams_cmd["BMV_Corrected"] <= 0.60)
    & (gaia_zams_cmd["DeltaFromZAMS"].between(-1.5, 0.8))
].copy()

if len(gaia_turnoff_candidates) < 3:
    gaia_estimated_age_gyr = np.nan
    gaia_turnoff_mv = np.nan
    gaia_turnoff_bmv = np.nan
    gaia_turnoff_mass = np.nan
    print(
        "Age not estimated reliably: the Gaia-member CMD contains too few "
        "plausible blue main-sequence/turnoff stars."
    )
else:
    # SAME AS SECTION 17
    gaia_turnoff_mv = gaia_turnoff_candidates["Absolute_V"].quantile(0.20)
    gaia_turnoff_bmv = gaia_turnoff_candidates.loc[
        (
            gaia_turnoff_candidates["Absolute_V"] - gaia_turnoff_mv
        ).abs().idxmin(),
        "BMV_Corrected",
    ]

    M_V_SUN = 4.83
    gaia_turnoff_luminosity = 10 ** (
        0.4 * (M_V_SUN - gaia_turnoff_mv)
    )
    gaia_turnoff_mass = gaia_turnoff_luminosity ** (1.0 / 3.5)
    gaia_estimated_age_gyr = 10.0 * gaia_turnoff_mass ** (-2.5)

    print(f"Approximate turnoff (B-V)_0:     {gaia_turnoff_bmv:.2f}")
    print(f"Approximate turnoff M_V:         {gaia_turnoff_mv:.2f}")
    print(f"Approximate turnoff mass:        {gaia_turnoff_mass:.2f} solar masses")
    print(f"Heuristic cluster age:           {gaia_estimated_age_gyr:.2f} Gyr")


### 16.7.2 Gaia-member CMD with the Section 17 ZAMS/turnoff result


In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(
    gaia_cmd["BMV_Corrected"],
    gaia_cmd["V_Corrected"],
    s=26,
    alpha=0.7,
    label="Gaia probable members",
)
plt.plot(
    zams["BMV_Corrected"],
    zams["M_V"] + gaia_distance_modulus,
    linewidth=2.5,
    label=f"Shifted ZAMS (mu0 = {gaia_distance_modulus:.2f})",
)

if np.isfinite(gaia_turnoff_bmv) and np.isfinite(gaia_turnoff_mv):
    plt.scatter(
        [gaia_turnoff_bmv],
        [gaia_turnoff_mv + gaia_distance_modulus],
        s=90,
        marker="*",
        label=f"Approx. turnoff ({gaia_turnoff_bmv:.2f}, {gaia_turnoff_mv:.2f})",
    )

plt.xlabel(r"Corrected color $(B-V)_0$")
plt.ylabel(r"Corrected magnitude $V_0$")
plt.title("NGC 2477 Gaia-Member CMD with ZAMS Overlay")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


### 16.7.3 Final Gaia-member-only age and distance estimate


In [ ]:
print("NGC 2477 — Gaia probable members only")
print("---------------------------------------")
print(f"Estimated age:      {gaia_estimated_age_gyr:.2f} Gyr")
print(f"Estimated distance: {gaia_distance_pc:,.0f} pc")
print(f"                    {gaia_distance_ly:,.0f} light-years")
print(f"Distance modulus:   {gaia_distance_modulus:.2f} mag")
print()
print("Age and distance use the same methodology and parameters as Section 17.")


# 17. Gaia Membership After Color-Color Screening

This chapter repeats the Gaia membership workflow using the photometric catalog after the B−V / V−R color-color screening.

The starting dataset is `OUTPUT_CALCED`, which is rewritten earlier after photometric outlier candidates are removed.

The purpose is to test whether combining color-color screening with Gaia astrometric membership filtering improves the final CMD, age, and distance estimate.


## 17.1 Load the post-screen photometric catalog


In [ ]:
screened_photometry = pd.read_csv(OUTPUT_CALCED).copy()

required_cols = ["StarID", "V_x_centroid", "V_y_centroid", "B_mag", "V_mag"]
missing_cols = [c for c in required_cols if c not in screened_photometry.columns]
if missing_cols:
    raise KeyError(f"OUTPUT_CALCED is missing required columns: {missing_cols}")

print(f"Post color-color screen stars loaded: {len(screened_photometry):,}")
display(screened_photometry.head())


## 17.2 Convert screened stars to RA/Dec and cross-match with Gaia


In [ ]:
# Use native V-image pixel positions directly for WCS conversion.
v_x_data = screened_photometry["V_x_centroid"].to_numpy(dtype=float)
v_y_data = screened_photometry["V_y_centroid"].to_numpy(dtype=float)

screened_sky = astrometric_wcs.pixel_to_world(v_x_data, v_y_data)
screened_photometry["RA_deg"] = screened_sky.ra.deg
screened_photometry["Dec_deg"] = screened_sky.dec.deg

gaia_sky = SkyCoord(
    ra=gaia_field["ra"].to_numpy(dtype=float) * u.deg,
    dec=gaia_field["dec"].to_numpy(dtype=float) * u.deg,
    frame="icrs",
)

idx, sep2d, _ = screened_sky.match_to_catalog_sky(gaia_sky)
screened_matched = sep2d <= GAIA_MATCH_RADIUS_ARCSEC * u.arcsec

screened_photometry["GaiaMatched"] = screened_matched
screened_photometry["GaiaMatchSepArcsec"] = sep2d.arcsec

gaia_columns = [
    "source_id", "ra", "dec",
    "parallax", "parallax_error",
    "pmra", "pmra_error",
    "pmdec", "pmdec_error",
    "ruwe", "astrometric_params_solved",
    "visibility_periods_used",
    "phot_g_mean_mag", "phot_bp_mean_mag",
    "phot_rp_mean_mag", "bp_rp",
]

for col in gaia_columns:
    screened_photometry[f"Gaia_{col}"] = np.nan

matched_rows = np.where(screened_matched)[0]
matched_gaia_rows = idx[screened_matched]

for col in gaia_columns:
    vals = gaia_field.iloc[matched_gaia_rows][col].to_numpy()
    screened_photometry.loc[
        screened_photometry.index[matched_rows],
        f"Gaia_{col}"
    ] = vals

print(
    f"Gaia matches after color-color screen: "
    f"{screened_matched.sum():,} / {len(screened_photometry):,}"
)


## 17.3 Apply the same Gaia sigma-cut membership criteria


In [ ]:
screened_has_valid_gaia = (
    screened_photometry["GaiaMatched"].fillna(False)
    & np.isfinite(screened_photometry["Gaia_parallax"])
    & np.isfinite(screened_photometry["Gaia_pmra"])
    & np.isfinite(screened_photometry["Gaia_pmdec"])
)

screened_photometry["GaiaParallaxSigmaOffset"] = np.nan
screened_photometry["GaiaPmraSigmaOffset"] = np.nan
screened_photometry["GaiaPmdecSigmaOffset"] = np.nan

valid_idx = screened_photometry.index[screened_has_valid_gaia]

screened_photometry.loc[valid_idx, "GaiaParallaxSigmaOffset"] = (
    np.abs(screened_photometry.loc[valid_idx, "Gaia_parallax"] - NGC2477_PARALLAX_CENTER)
    / NGC2477_PARALLAX_SIGMA
)
screened_photometry.loc[valid_idx, "GaiaPmraSigmaOffset"] = (
    np.abs(screened_photometry.loc[valid_idx, "Gaia_pmra"] - NGC2477_PMRA_CENTER)
    / NGC2477_PMRA_SIGMA
)
screened_photometry.loc[valid_idx, "GaiaPmdecSigmaOffset"] = (
    np.abs(screened_photometry.loc[valid_idx, "Gaia_pmdec"] - NGC2477_PMDEC_CENTER)
    / NGC2477_PMDEC_SIGMA
)

screened_photometry["GaiaProbableMember"] = (
    screened_has_valid_gaia
    & (screened_photometry["GaiaParallaxSigmaOffset"] <= GAIA_MEMBERSHIP_SIGMA)
    & (screened_photometry["GaiaPmraSigmaOffset"] <= GAIA_MEMBERSHIP_SIGMA)
    & (screened_photometry["GaiaPmdecSigmaOffset"] <= GAIA_MEMBERSHIP_SIGMA)
)

screened_photometry["GaiaMembershipStatus"] = "Non-member"
screened_photometry.loc[
    screened_photometry["GaiaProbableMember"],
    "GaiaMembershipStatus"
] = "Probable member"

print(
    f"Probable members after color-color + Gaia filtering: "
    f"{screened_photometry['GaiaProbableMember'].sum():,} / "
    f"{len(screened_photometry):,}"
)


## 17.4 Keep color-screened + Gaia probable members only


In [ ]:
screened_gaia_members = screened_photometry[
    screened_photometry["GaiaProbableMember"]
].copy()

SCREENED_GAIA_MEMBERS_OUTPUT = str(
    Path(OUTPUT_CALCED).with_name("NGC2477_color_screen_gaia_probable_members.csv")
)

screened_gaia_members.to_csv(SCREENED_GAIA_MEMBERS_OUTPUT, index=False)

print(f"Saved members: {len(screened_gaia_members):,}")
print(f"Output file: {SCREENED_GAIA_MEMBERS_OUTPUT}")


## 17.5 Estimate distance and age using color-screened Gaia members

This calculation uses the **same methodology and parameter values as Section 17 and Section 18.11**, changing only the input sample to stars that pass both the color-color screen and Gaia membership screen.


In [ ]:
screened_gaia_cmd = screened_gaia_members.copy()

screened_gaia_cmd["BMV"] = (
    screened_gaia_cmd["B_mag"] - screened_gaia_cmd["V_mag"]
)
screened_gaia_cmd["BMV_Corrected"] = (
    screened_gaia_cmd["BMV"] - BMV_ReddenAdjustment
)
screened_gaia_cmd["V_Corrected"] = (
    screened_gaia_cmd["V_mag"] - V_ReddenAdjustment
)

screened_gaia_cmd = screened_gaia_cmd[
    screened_gaia_cmd["BMV_Corrected"].between(-0.4, 2.0)
    & np.isfinite(screened_gaia_cmd["V_Corrected"])
].copy()

if len(screened_gaia_cmd) < 10:
    raise RuntimeError(
        "Too few color-screened Gaia members remain for age/distance estimation."
    )

screened_zams_cmd = screened_gaia_cmd[
    screened_gaia_cmd["BMV_Corrected"].between(
        zams["BMV_Corrected"].min(),
        zams["BMV_Corrected"].max()
    )
].copy()

if len(screened_zams_cmd) < 10:
    raise RuntimeError(
        "Too few color-screened Gaia members lie within the ZAMS color range."
    )

screened_zams_cmd["ZAMS_MV"] = np.interp(
    screened_zams_cmd["BMV_Corrected"],
    zams["BMV_Corrected"],
    zams["M_V"],
)
screened_zams_cmd["IndividualDistanceModulus"] = (
    screened_zams_cmd["V_Corrected"] - screened_zams_cmd["ZAMS_MV"]
)

# SAME AS SECTION 17 AND SECTION 18.11
DISTANCE_MODULUS_QUANTILE = 0.70
screened_gaia_fitted_distance_modulus = screened_zams_cmd[
    "IndividualDistanceModulus"
].quantile(DISTANCE_MODULUS_QUANTILE)

MANUAL_DISTANCE_MODULUS = None
screened_gaia_distance_modulus = (
    screened_gaia_fitted_distance_modulus
    if MANUAL_DISTANCE_MODULUS is None
    else float(MANUAL_DISTANCE_MODULUS)
)

screened_gaia_distance_pc = 10 ** (
    (screened_gaia_distance_modulus + 5.0) / 5.0
)
screened_gaia_distance_ly = screened_gaia_distance_pc * 3.26156

screened_zams_cmd["Absolute_V"] = (
    screened_zams_cmd["V_Corrected"] - screened_gaia_distance_modulus
)
screened_zams_cmd["DeltaFromZAMS"] = (
    screened_zams_cmd["Absolute_V"] - screened_zams_cmd["ZAMS_MV"]
)

# SAME AS SECTION 17 AND SECTION 18.11
screened_turnoff_candidates = screened_zams_cmd[
    (screened_zams_cmd["BMV_Corrected"] <= 0.60)
    & (screened_zams_cmd["DeltaFromZAMS"].between(-1.5, 0.8))
].copy()

if len(screened_turnoff_candidates) < 3:
    screened_gaia_estimated_age_gyr = np.nan
    screened_gaia_turnoff_mv = np.nan
    screened_gaia_turnoff_bmv = np.nan
    screened_gaia_turnoff_mass = np.nan
    print(
        "Age not estimated reliably: too few plausible blue "
        "main-sequence/turnoff stars remain."
    )
else:
    screened_gaia_turnoff_mv = screened_turnoff_candidates[
        "Absolute_V"
    ].quantile(0.20)

    screened_gaia_turnoff_bmv = screened_turnoff_candidates.loc[
        (
            screened_turnoff_candidates["Absolute_V"]
            - screened_gaia_turnoff_mv
        ).abs().idxmin(),
        "BMV_Corrected",
    ]

    M_V_SUN = 4.83
    screened_gaia_turnoff_luminosity = 10 ** (
        0.4 * (M_V_SUN - screened_gaia_turnoff_mv)
    )
    screened_gaia_turnoff_mass = (
        screened_gaia_turnoff_luminosity ** (1.0 / 3.5)
    )
    screened_gaia_estimated_age_gyr = (
        10.0 * screened_gaia_turnoff_mass ** (-2.5)
    )

print("NGC 2477 — Color-color screen + Gaia probable members")
print("-------------------------------------------------------")
print(f"Estimated age:      {screened_gaia_estimated_age_gyr:.2f} Gyr")
print(f"Estimated distance: {screened_gaia_distance_pc:,.0f} pc")
print(f"                    {screened_gaia_distance_ly:,.0f} light-years")
print(f"Distance modulus:   {screened_gaia_distance_modulus:.2f} mag")

if np.isfinite(screened_gaia_turnoff_bmv):
    print(f"Approximate turnoff (B-V)_0: {screened_gaia_turnoff_bmv:.2f}")
    print(f"Approximate turnoff M_V:     {screened_gaia_turnoff_mv:.2f}")
    print(f"Approximate turnoff mass:    {screened_gaia_turnoff_mass:.2f} solar masses")


## 17.6 CMD for color-screened Gaia probable members


In [ ]:
plt.figure(figsize=(10, 8))

plt.scatter(
    screened_gaia_cmd["BMV_Corrected"],
    screened_gaia_cmd["V_Corrected"],
    s=28,
    alpha=0.8,
    label="Color-screened + Gaia members",
)

plt.plot(
    zams["BMV_Corrected"],
    zams["M_V"] + screened_gaia_distance_modulus,
    linewidth=2.5,
    label=f"Shifted ZAMS (mu0 = {screened_gaia_distance_modulus:.2f})",
)

if (
    np.isfinite(screened_gaia_turnoff_bmv)
    and np.isfinite(screened_gaia_turnoff_mv)
):
    plt.scatter(
        [screened_gaia_turnoff_bmv],
        [screened_gaia_turnoff_mv + screened_gaia_distance_modulus],
        s=90,
        marker="*",
        label=(
            f"Approx. turnoff "
            f"({screened_gaia_turnoff_bmv:.2f}, {screened_gaia_turnoff_mv:.2f})"
        ),
    )

plt.xlabel(r"Corrected color $(B-V)_0$")
plt.ylabel(r"Corrected magnitude $V_0$")
plt.title("NGC 2477 CMD — Color-Color Screen + Gaia Membership")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
